# Private cross-repository grader diagnostic23

Use Accelerator None, Internet off, compatible CPython3.12 Linux x86_64, official data and wheelhouse. These are the same four already exposed panel22 cases and eight clean/reference checks. Panel22 completed with no qualified cases; this attempt retains private grader errors to diagnose them. The official verifier, setup and task order are unchanged. No model, generated repairs or new selection.

Private diagnostic outputs contain validation plaintext and must never become agent inputs or public resource assets. Full reference patches and raw task records are not exported. No holdout, leaderboard or device result is claimed. USB testing remains deferred.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, base64, zipfile, io, time, inspect
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_DECLARED_EXPERIMENT = True
SESSION_STARTED_MONOTONIC = time.monotonic()
MODE = 'cpu_controls'
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not IS_KAGGLE:
    print('Definitions only: local execution performs no official inference, fixture grading or installation')


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifacts, manifest = [], []
    for artifact in sorted(run_root.rglob('*')):
        resolved = artifact.resolve()
        if artifact.is_symlink() or not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact is a link or escaped its directory')
        name = artifact.relative_to(run_root).as_posix()
        if name == 'evidence_bundle.zip' or name.startswith('evidence_checkpoints/'):
            continue
        if not artifact.is_file():
            continue
        if name == 'artifact_manifest.json':
            raise ValueError('artifact_manifest.json is reserved for the exported manifest')
        # Read once. A live log may grow, but its archived bytes and hash must agree.
        payload = artifact.read_bytes()
        artifacts.append((name, payload))
        manifest.append({'path': name, 'bytes': len(payload),
                         'sha256': hashlib.sha256(payload).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, payload in artifacts:
            archive.writestr(name, payload)
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    digest = hashlib.sha256(bundle_bytes).hexdigest()
    checkpoints = run_root / 'evidence_checkpoints'
    checkpoints.mkdir(exist_ok=True)
    checkpoint = checkpoints / (digest + '.zip')
    if checkpoint.exists():
        if checkpoint.read_bytes() != bundle_bytes:
            raise ValueError('Existing checkpoint differs from its content hash')
    else:
        with checkpoint.open('xb') as stream:
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=run_root, prefix='.evidence_bundle.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, run_root / 'evidence_bundle.zip')
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    report = {'sha256': digest, 'bytes': len(bundle_bytes), 'file_count': len(manifest),
              'checkpoint_path': checkpoint.relative_to(run_root).as_posix(),
              'snapshot_is_atomic_across_all_files': False}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
PLAN = json.loads('{"created_utc": "2026-10-05T10:09:55.519503+00:00", "scope": "private_diagnostic_of_same_exposed_panel22_not_new_selection_or_agent_evaluation", "tasks": [{"instance_id": "requests_6644", "repo": "psf/requests", "base_commit": "382fc2c0c6c0ef0874bc65bc1175f97c073e5086"}, {"instance_id": "fastapi_15023", "repo": "fastapi/fastapi", "base_commit": "c0836dc1b7f4ba91e9563c8c4ccd0ebe655bc914"}, {"instance_id": "requests_7433", "repo": "psf/requests", "base_commit": "0b401c76b6e80a4eecf3c690085b2553f6e261ca"}, {"instance_id": "fastapi_14512", "repo": "fastapi/fastapi", "base_commit": "1fcec88ad2df4541b6a81c4786c4d039dc745e1e"}], "tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "guide_sha256": "3d6e57a13234cb4e783ba24eaab486459af76e0923ea4c6607cd41cc8961bbbb", "expected_core_packages": {"swegemma": "0.2.7", "adk-submission": "0.2.12", "adk-eval-core": "0.1.0"}, "expected_verify_task_sha256": "98600fd6066a4dbd472490cb194b839a3a633afea9b99256cc081ea77beac0d3", "selected_problem_only_input_sha256": "4dbfcbb2151487854269975eaed936370d7ac9a003b48523a2b2362430fc23f4", "selection": {"salt": "20261005_crossrepo_public_controls_v1", "repositories": ["psf/requests", "fastapi/fastapi"], "cases_per_repository": 2, "method": "First two eligible IDs by SHA256(salt + colon + instance_id), interleaved by repository", "ranked_eligible_ids": {"psf/requests": ["requests_6644", "requests_7433", "requests_7315", "requests_7427", "requests_7502", "requests_7205", "requests_7505", "requests_7328", "requests_6589", "requests_6629", "requests_6757", "requests_7309"], "fastapi/fastapi": ["fastapi_15023", "fastapi_14512", "fastapi_15030", "fastapi_14482", "fastapi_14455", "fastapi_14463", "fastapi_5077", "fastapi_14371", "fastapi_11194", "fastapi_14616", "fastapi_9753", "fastapi_13920", "fastapi_15745", "fastapi_14301", "fastapi_14258", "fastapi_14356", "fastapi_15785", "fastapi_14303", "fastapi_14479", "fastapi_14609", "fastapi_12942", "fastapi_14492", "fastapi_15589", "fastapi_14306", "fastapi_15280", "fastapi_14361", "fastapi_13786", "fastapi_15800", "fastapi_13207", "fastapi_13713", "fastapi_14186", "fastapi_14458", "fastapi_15763", "fastapi_14794", "fastapi_14873", "fastapi_14266", "fastapi_14448", "fastapi_14246", "fastapi_14099", "fastapi_14953", "fastapi_9425", "fastapi_14605", "fastapi_14485", "fastapi_14978", "fastapi_14986", "fastapi_14349", "fastapi_5624", "fastapi_14791", "fastapi_14372", "fastapi_14459", "fastapi_14583", "fastapi_14360", "fastapi_14962", "fastapi_14419", "fastapi_13537", "fastapi_14786", "fastapi_14297", "fastapi_15588", "fastapi_11355", "fastapi_14964", "fastapi_9555", "fastapi_14430", "fastapi_14487", "fastapi_14851"]}, "excluded_prior_retained_ids": ["fastapi_14077", "fastapi_14262", "fastapi_15661", "requests_6592", "rich_2725", "rich_3067", "rich_4079"], "prior_retained_id_audit": {"reports/kaggle_runs/20261003_development/analysis.json": {"sha256": "dc6ed351014b13d643641af9f2443f9fd58cb1189d0562488c2f1477003321bc", "retained_ids": ["requests_6592", "rich_3067"]}, "reports/kaggle_runs/20261003_development/run_provenance.json": {"sha256": "33ccc3932deb4765cf502db5b41ddf4de08e70b569242cfb4853e28aee64fb49", "retained_ids": []}, "reports/kaggle_runs/20261003_fixture_integrity/analysis.json": {"sha256": "1c9198909280cf662f34d2916a392424fdb0a0c6fbf8618ee177aa2cd19fdd42", "retained_ids": ["requests_6592", "rich_3067"]}, "reports/kaggle_runs/20261003_fixture_integrity/run_provenance.json": {"sha256": "56a8218f2d6bbe750883e47dd6ff838c35cafcd59ecbc306eb2652393b78adb7", "retained_ids": []}, "reports/kaggle_runs/20261003_smoke/analysis.json": {"sha256": "3e10488a53515070bba300572fe78ce012dc67d9f70ca3cff4eab92ced50a07c", "retained_ids": []}, "reports/kaggle_runs/20261004_eval_config_source_probe/run_provenance.json": {"sha256": "478991ee605eaaff170f32eb11d3fefc34101b7d959fdda897c0c54f2c38c33e", "retained_ids": []}, "reports/kaggle_runs/20261004_fixture_dependencies/analysis.json": {"sha256": "938a5431830e14aa09cbc1f1b42e0e28fc7334111a5493c6e4df061391e287b9", "retained_ids": ["requests_6592", "rich_3067"]}, "reports/kaggle_runs/20261004_fixture_dependencies/run_provenance.json": {"sha256": "214f0ef726971537555a21b5baf6f41828c382cd5816d89ea529effc11630153", "retained_ids": []}, "reports/kaggle_runs/20261004_legacy_redirect/analysis.json": {"sha256": "9b01f19ceea8cd7f37a0fefef58d1f501490469846db6038f7416e825e7811fb", "retained_ids": ["requests_6592", "rich_3067"]}, "reports/kaggle_runs/20261004_legacy_redirect/run_provenance.json": {"sha256": "e767130b2a96801f984d64fa428ee952e29d77d2f76b4e642f1ba9d10ab97513", "retained_ids": []}, "reports/kaggle_runs/20261004_mobile_source_audit/run_provenance.json": {"sha256": "f2c5f01d49fd21366625a56f3590ac89e9e109e1ad31c186cd44bf401123bc4f", "retained_ids": []}, "reports/kaggle_runs/20261004_mobile_source_audit_retry1/run_provenance.json": {"sha256": "b5940bfe9c82d47adca07495914078c2f69366f8d886664460ae0af431e1d94a", "retained_ids": []}, "reports/kaggle_runs/20261004_mobile_source_audit_retry2/run_provenance.json": {"sha256": "84c6058f9f5ee7ae486ce24b2e6321b932db770e8f22cf26572cd86dd7ade1aa", "retained_ids": []}, "reports/kaggle_runs/20261004_paper_extracted_replay/run_provenance.json": {"sha256": "dd4d7ca9682eafb74ece9674813dfd20101fc3aa480df4db0f339cde9f6deec1", "retained_ids": []}, "reports/kaggle_runs/20261004_public_fixture_controls/run_provenance.json": {"sha256": "780b6a7af49b62a8751d85644760c357b44aaad56593d7da17791778e1f39803", "retained_ids": ["fastapi_14077", "fastapi_14262", "rich_2725", "rich_4079"]}, "reports/kaggle_runs/20261004_rich_fixture_qualified_cap8192/analysis.json": {"sha256": "1f0756d9bd1d3ffee1bd7bde36e2c9223a8737131ea4dfbdab74a43096869b8c", "retained_ids": ["rich_2725", "rich_4079"]}, "reports/kaggle_runs/20261004_rich_fixture_qualified_cap8192/run_provenance.json": {"sha256": "f82909d44794652fb227b2d2c2a7c6429f760a055314032fcd5a4a1e88d47d67", "retained_ids": ["rich_2725", "rich_4079"]}, "reports/kaggle_runs/20261004_source_only_v3/analysis.json": {"sha256": "a7f32ac83c4019a07b8b0027219bb86e6b1778db521285e6d6875e0bce1a0d4e", "retained_ids": []}, "reports/kaggle_runs/20261004_source_only_v3/run_provenance.json": {"sha256": "a70f4a92073308fd5700b58dd3962a9bc94af4ffa2c3f20aa94643f132ecfda9", "retained_ids": []}, "reports/kaggle_runs/20261004_source_only_v3_batch/run_provenance.json": {"sha256": "153c78849c59f8768d813d0efe1aa6b2821b88da68ff40cd7a922e6e3644b46c", "retained_ids": []}, "reports/kaggle_runs/20261004_source_only_v3_batch_retry1/analysis.json": {"sha256": "fb8b603bf5bc9d94fd37b26a78e86e9b25fcf58c53a7a88689b90a444a58b2e7", "retained_ids": ["rich_3067"]}, "reports/kaggle_runs/20261004_source_only_v3_batch_retry1/run_provenance.json": {"sha256": "176b9f73a83a4236776108c888b7333cd7695e3c04056622c3e9882f388b399e", "retained_ids": []}, "reports/kaggle_runs/20261005_source_budget_compaction5_transfer_revision1/run_provenance.json": {"sha256": "cc32716ba60900d7ce29a9a1fae40bbba63c5106a88d65a2922e1ffdcbfcaafa", "retained_ids": ["rich_2725", "rich_4079"]}}, "source_problem_only_views": {"benchmarks/competition/studies/20261003_v1/fold_00/agent/calibration.jsonl": "a93764c98ccc65851992820c5e7f5cac1170d54ed09cc2d3355098a59d330a56", "benchmarks/competition/studies/20261003_v1/fold_00/agent/fit.jsonl": "4899aa84910847719ef6f2424b295e571ba0e78dc1eb406720c4735d06f352c4", "benchmarks/competition/studies/20261003_v1/fold_00/agent/held_out.jsonl": "cbac69ce7c6b8d687cbcb81ab126cc46427034a6f5d54cab5f1ac95397e8039f", "benchmarks/competition/studies/20261003_v1/fold_01/agent/calibration.jsonl": "12b1c0c5699e40980f08ba3add3c11455454f743461df8710a6c7ff60e79662b", "benchmarks/competition/studies/20261003_v1/fold_01/agent/fit.jsonl": "a2827985361d35fc09f0c2201e6e7c647d4469ee1ee9afb4185c16d1134b2954", "benchmarks/competition/studies/20261003_v1/fold_01/agent/held_out.jsonl": "a93764c98ccc65851992820c5e7f5cac1170d54ed09cc2d3355098a59d330a56", "benchmarks/competition/studies/20261003_v1/fold_02/agent/calibration.jsonl": "5402589d1a04aaf8e0420d0d14d9aaad82f8a2226862f533e7cbf2042b39766f", "benchmarks/competition/studies/20261003_v1/fold_02/agent/fit.jsonl": "64381cb894671d75deb7040e0ab9c0cf9830f48c0c4de002229fc724f6ad7203", "benchmarks/competition/studies/20261003_v1/fold_02/agent/held_out.jsonl": "12b1c0c5699e40980f08ba3add3c11455454f743461df8710a6c7ff60e79662b", "benchmarks/competition/studies/20261003_v1/fold_03/agent/calibration.jsonl": "cbac69ce7c6b8d687cbcb81ab126cc46427034a6f5d54cab5f1ac95397e8039f", "benchmarks/competition/studies/20261003_v1/fold_03/agent/fit.jsonl": "d684c2c79d404d2d6340c3d9603c008618027d3c828ad7b6330f06dffa942256", "benchmarks/competition/studies/20261003_v1/fold_03/agent/held_out.jsonl": "5402589d1a04aaf8e0420d0d14d9aaad82f8a2226862f533e7cbf2042b39766f"}, "issue_text_or_reference_or_outcome_used_to_rank": false, "reference_views_read_by_preparation": false}, "known_development_results": {"parent18_actual_analysis_sha256": "1f0756d9bd1d3ffee1bd7bde36e2c9223a8737131ea4dfbdab74a43096869b8c", "parent18_result": "source1/2_path1/2_identical_successful_patch_no_gain", "transfer21_provenance_sha256": "cc32716ba60900d7ce29a9a1fae40bbba63c5106a88d65a2922e1ffdcbfcaafa", "transfer21_last_retained_state": "queued", "these_four_case_model_outcomes_available_at_freeze": false}, "limits": {"timeout_seconds": 300, "max_time_minutes": 20, "max_tool_calls": 50, "max_turns": 100}, "timeout_semantics": "Per sandbox command including verifier pytest; no whole-verifier deadline", "fixture_controls": {"accelerator": "none", "model_inference": false, "assigned_checks": 8, "conditions": ["clean_base", "official_reference_patch"], "reference_access": "Independent CPU verifier only; full reference patches stay in memory. Private test output and grader error text are retained solely for environment diagnosis."}, "arms": {}, "schedule": [], "future_model_gate": "Reconcile21 before new GPU spending; a separate frozen plan must bind any arms and model assignments", "future_control_selection_rule": "Choose first metadata-ranked clean_base_nonpassing_reference_verified case per repository for a prospective four-slot paired development diagnostic; if any repository has no qualified case, report insufficient coverage without replacement", "stopping_rule": "Same eight checks and order; save actual failure and halt on raised infrastructure exception. No refill, retry, model inference or environment changes.", "research_assistant_public_pool_reference_exposure_recorded": true, "new_problem_text_preparation_exposure": "Only existing five-field views; earlier metadata inspection printed several Requests problem statements, including7433; no evaluator reference text read here", "holdout_or_confirmatory_claim_allowed": false, "agent_generated_repairs": null, "official_leaderboard_score": null, "competition_entry_submitted": false, "USB_testing": "deferred_at_user_request", "parent22_plan_sha256": "a9baad2efaac6a89d0bc416e703fab1e173250488e2abf848123d304a05924f5", "parent22_analysis_sha256": "e4e55870c003958b4323b3d315546a6eaa18f4f1fabc6c8fd82db411bfd2f295", "parent22_known_classifications": [{"task_id": "requests_6644", "classification": "clean_base_already_passes"}, {"task_id": "fastapi_15023", "classification": "official_reference_not_verified"}, {"task_id": "requests_7433", "classification": "official_reference_not_verified"}, {"task_id": "fastapi_14512", "classification": "official_reference_not_verified"}], "transfer21_completed_provenance_sha256": "fda6ca611704d3b6f8e84a7e3fff0df9df5384f7d47d3d066938dbfea465184a", "transfer21_known_result": "1_of_2_both_50_tools_no_gain_or_selection", "new_cases_selected": false, "environment_modification_from_parent22": false, "private_grader_text_retained": true, "reference_patch_exported": false, "agent_or_public_resource_input": false, "diagnostic_followup_rule": "Review actual grader errors before declaring a separate environment repair; do not mutate these frozen controls or treat their reference outcomes as model repairs"}')
PLAN_BYTES_BASE64 = 'ewogICJjcmVhdGVkX3V0YyI6ICIyMDI2LTEwLTA1VDEwOjA5OjU1LjUxOTUwMyswMDowMCIsCiAgInNjb3BlIjogInByaXZhdGVfZGlhZ25vc3RpY19vZl9zYW1lX2V4cG9zZWRfcGFuZWwyMl9ub3RfbmV3X3NlbGVjdGlvbl9vcl9hZ2VudF9ldmFsdWF0aW9uIiwKICAidGFza3MiOiBbCiAgICB7CiAgICAgICJpbnN0YW5jZV9pZCI6ICJyZXF1ZXN0c182NjQ0IiwKICAgICAgInJlcG8iOiAicHNmL3JlcXVlc3RzIiwKICAgICAgImJhc2VfY29tbWl0IjogIjM4MmZjMmMwYzZjMGVmMDg3NGJjNjViYzExNzVmOTdjMDczZTUwODYiCiAgICB9LAogICAgewogICAgICAiaW5zdGFuY2VfaWQiOiAiZmFzdGFwaV8xNTAyMyIsCiAgICAgICJyZXBvIjogImZhc3RhcGkvZmFzdGFwaSIsCiAgICAgICJiYXNlX2NvbW1pdCI6ICJjMDgzNmRjMWI3ZjRiYTkxZTk1NjNjOGM0Y2NkMGViZTY1NWJjOTE0IgogICAgfSwKICAgIHsKICAgICAgImluc3RhbmNlX2lkIjogInJlcXVlc3RzXzc0MzMiLAogICAgICAicmVwbyI6ICJwc2YvcmVxdWVzdHMiLAogICAgICAiYmFzZV9jb21taXQiOiAiMGI0MDFjNzZiNmU4MGE0ZWVjZjNjNjkwMDg1YjI1NTNmNmUyNjFjYSIKICAgIH0sCiAgICB7CiAgICAgICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE0NTEyIiwKICAgICAgInJlcG8iOiAiZmFzdGFwaS9mYXN0YXBpIiwKICAgICAgImJhc2VfY29tbWl0IjogIjFmY2VjODhhZDJkZjQ1NDFiNmE4MWM0Nzg2YzRkMDM5ZGM3NDVlMWUiCiAgICB9CiAgXSwKICAidGFza3Nfc2hhMjU2IjogImU0YjNmZDYwZjY5ZGJjMmI5MjEzZTU0ZWViOTYzNmRiNzhhZWZlOTJjMWQwNjI2OWQ3M2Q5ZjVmOGYzYzhhZDYiLAogICJndWlkZV9zaGEyNTYiOiAiM2Q2ZTU3YTEzMjM0Y2I0ZTc4M2JhMjRlYWFiNDg2NDU5YWY3NmUwOTIzZWE0YzY2MDdjZDQxY2M4OTYxYmJiYiIsCiAgImV4cGVjdGVkX2NvcmVfcGFja2FnZXMiOiB7CiAgICAic3dlZ2VtbWEiOiAiMC4yLjciLAogICAgImFkay1zdWJtaXNzaW9uIjogIjAuMi4xMiIsCiAgICAiYWRrLWV2YWwtY29yZSI6ICIwLjEuMCIKICB9LAogICJleHBlY3RlZF92ZXJpZnlfdGFza19zaGEyNTYiOiAiOTg2MDBmZDYwNjZhNGRiZDQ3MjQ5MGNiMTk0YjgzOWEzYTYzM2FmZWE5Yjk5MjU2Y2MwODFlYTc3YmVhYzBkMyIsCiAgInNlbGVjdGVkX3Byb2JsZW1fb25seV9pbnB1dF9zaGEyNTYiOiAiNGRiZmNiYjIxNTE0ODc4NTQyNjk5NzVlYWVkOTM2MzcwZDdhYzlhMDAzYjQ4NTIzYTJiMjM2MjQzMGZjMjNmNCIsCiAgInNlbGVjdGlvbiI6IHsKICAgICJzYWx0IjogIjIwMjYxMDA1X2Nyb3NzcmVwb19wdWJsaWNfY29udHJvbHNfdjEiLAogICAgInJlcG9zaXRvcmllcyI6IFsKICAgICAgInBzZi9yZXF1ZXN0cyIsCiAgICAgICJmYXN0YXBpL2Zhc3RhcGkiCiAgICBdLAogICAgImNhc2VzX3Blcl9yZXBvc2l0b3J5IjogMiwKICAgICJtZXRob2QiOiAiRmlyc3QgdHdvIGVsaWdpYmxlIElEcyBieSBTSEEyNTYoc2FsdCArIGNvbG9uICsgaW5zdGFuY2VfaWQpLCBpbnRlcmxlYXZlZCBieSByZXBvc2l0b3J5IiwKICAgICJyYW5rZWRfZWxpZ2libGVfaWRzIjogewogICAgICAicHNmL3JlcXVlc3RzIjogWwogICAgICAgICJyZXF1ZXN0c182NjQ0IiwKICAgICAgICAicmVxdWVzdHNfNzQzMyIsCiAgICAgICAgInJlcXVlc3RzXzczMTUiLAogICAgICAgICJyZXF1ZXN0c183NDI3IiwKICAgICAgICAicmVxdWVzdHNfNzUwMiIsCiAgICAgICAgInJlcXVlc3RzXzcyMDUiLAogICAgICAgICJyZXF1ZXN0c183NTA1IiwKICAgICAgICAicmVxdWVzdHNfNzMyOCIsCiAgICAgICAgInJlcXVlc3RzXzY1ODkiLAogICAgICAgICJyZXF1ZXN0c182NjI5IiwKICAgICAgICAicmVxdWVzdHNfNjc1NyIsCiAgICAgICAgInJlcXVlc3RzXzczMDkiCiAgICAgIF0sCiAgICAgICJmYXN0YXBpL2Zhc3RhcGkiOiBbCiAgICAgICAgImZhc3RhcGlfMTUwMjMiLAogICAgICAgICJmYXN0YXBpXzE0NTEyIiwKICAgICAgICAiZmFzdGFwaV8xNTAzMCIsCiAgICAgICAgImZhc3RhcGlfMTQ0ODIiLAogICAgICAgICJmYXN0YXBpXzE0NDU1IiwKICAgICAgICAiZmFzdGFwaV8xNDQ2MyIsCiAgICAgICAgImZhc3RhcGlfNTA3NyIsCiAgICAgICAgImZhc3RhcGlfMTQzNzEiLAogICAgICAgICJmYXN0YXBpXzExMTk0IiwKICAgICAgICAiZmFzdGFwaV8xNDYxNiIsCiAgICAgICAgImZhc3RhcGlfOTc1MyIsCiAgICAgICAgImZhc3RhcGlfMTM5MjAiLAogICAgICAgICJmYXN0YXBpXzE1NzQ1IiwKICAgICAgICAiZmFzdGFwaV8xNDMwMSIsCiAgICAgICAgImZhc3RhcGlfMTQyNTgiLAogICAgICAgICJmYXN0YXBpXzE0MzU2IiwKICAgICAgICAiZmFzdGFwaV8xNTc4NSIsCiAgICAgICAgImZhc3RhcGlfMTQzMDMiLAogICAgICAgICJmYXN0YXBpXzE0NDc5IiwKICAgICAgICAiZmFzdGFwaV8xNDYwOSIsCiAgICAgICAgImZhc3RhcGlfMTI5NDIiLAogICAgICAgICJmYXN0YXBpXzE0NDkyIiwKICAgICAgICAiZmFzdGFwaV8xNTU4OSIsCiAgICAgICAgImZhc3RhcGlfMTQzMDYiLAogICAgICAgICJmYXN0YXBpXzE1MjgwIiwKICAgICAgICAiZmFzdGFwaV8xNDM2MSIsCiAgICAgICAgImZhc3RhcGlfMTM3ODYiLAogICAgICAgICJmYXN0YXBpXzE1ODAwIiwKICAgICAgICAiZmFzdGFwaV8xMzIwNyIsCiAgICAgICAgImZhc3RhcGlfMTM3MTMiLAogICAgICAgICJmYXN0YXBpXzE0MTg2IiwKICAgICAgICAiZmFzdGFwaV8xNDQ1OCIsCiAgICAgICAgImZhc3RhcGlfMTU3NjMiLAogICAgICAgICJmYXN0YXBpXzE0Nzk0IiwKICAgICAgICAiZmFzdGFwaV8xNDg3MyIsCiAgICAgICAgImZhc3RhcGlfMTQyNjYiLAogICAgICAgICJmYXN0YXBpXzE0NDQ4IiwKICAgICAgICAiZmFzdGFwaV8xNDI0NiIsCiAgICAgICAgImZhc3RhcGlfMTQwOTkiLAogICAgICAgICJmYXN0YXBpXzE0OTUzIiwKICAgICAgICAiZmFzdGFwaV85NDI1IiwKICAgICAgICAiZmFzdGFwaV8xNDYwNSIsCiAgICAgICAgImZhc3RhcGlfMTQ0ODUiLAogICAgICAgICJmYXN0YXBpXzE0OTc4IiwKICAgICAgICAiZmFzdGFwaV8xNDk4NiIsCiAgICAgICAgImZhc3RhcGlfMTQzNDkiLAogICAgICAgICJmYXN0YXBpXzU2MjQiLAogICAgICAgICJmYXN0YXBpXzE0NzkxIiwKICAgICAgICAiZmFzdGFwaV8xNDM3MiIsCiAgICAgICAgImZhc3RhcGlfMTQ0NTkiLAogICAgICAgICJmYXN0YXBpXzE0NTgzIiwKICAgICAgICAiZmFzdGFwaV8xNDM2MCIsCiAgICAgICAgImZhc3RhcGlfMTQ5NjIiLAogICAgICAgICJmYXN0YXBpXzE0NDE5IiwKICAgICAgICAiZmFzdGFwaV8xMzUzNyIsCiAgICAgICAgImZhc3RhcGlfMTQ3ODYiLAogICAgICAgICJmYXN0YXBpXzE0Mjk3IiwKICAgICAgICAiZmFzdGFwaV8xNTU4OCIsCiAgICAgICAgImZhc3RhcGlfMTEzNTUiLAogICAgICAgICJmYXN0YXBpXzE0OTY0IiwKICAgICAgICAiZmFzdGFwaV85NTU1IiwKICAgICAgICAiZmFzdGFwaV8xNDQzMCIsCiAgICAgICAgImZhc3RhcGlfMTQ0ODciLAogICAgICAgICJmYXN0YXBpXzE0ODUxIgogICAgICBdCiAgICB9LAogICAgImV4Y2x1ZGVkX3ByaW9yX3JldGFpbmVkX2lkcyI6IFsKICAgICAgImZhc3RhcGlfMTQwNzciLAogICAgICAiZmFzdGFwaV8xNDI2MiIsCiAgICAgICJmYXN0YXBpXzE1NjYxIiwKICAgICAgInJlcXVlc3RzXzY1OTIiLAogICAgICAicmljaF8yNzI1IiwKICAgICAgInJpY2hfMzA2NyIsCiAgICAgICJyaWNoXzQwNzkiCiAgICBdLAogICAgInByaW9yX3JldGFpbmVkX2lkX2F1ZGl0IjogewogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwM19kZXZlbG9wbWVudC9hbmFseXNpcy5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiZGM2ZWQzNTEwMTRiMTNkNjQzNjQxYWY5ZjI0NDNmOWZkNThjYjExODlkMDU2MjQ4OGMyZjE0NzcwMDMzMjFiYyIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFsKICAgICAgICAgICJyZXF1ZXN0c182NTkyIiwKICAgICAgICAgICJyaWNoXzMwNjciCiAgICAgICAgXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwM19kZXZlbG9wbWVudC9ydW5fcHJvdmVuYW5jZS5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiMzNjY2MzOTMyZGViNDc2NWNmNTAyZGI1YjQxZGRmNGRlMDhlNzBiNTY5MjQyY2ZiNDg1M2UyOGFlZTY0ZmI0OSIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFtdCiAgICAgIH0sCiAgICAgICJyZXBvcnRzL2thZ2dsZV9ydW5zLzIwMjYxMDAzX2ZpeHR1cmVfaW50ZWdyaXR5L2FuYWx5c2lzLmpzb24iOiB7CiAgICAgICAgInNoYTI1NiI6ICIxYzkxOTg5MDkyODBjZjY2MmYzNGQyOTE2YTM5MjQyNGZkYjBhMGM2ZmJmODYxOGVlMTc3YWEyY2QxOWZkZDQyIiwKICAgICAgICAicmV0YWluZWRfaWRzIjogWwogICAgICAgICAgInJlcXVlc3RzXzY1OTIiLAogICAgICAgICAgInJpY2hfMzA2NyIKICAgICAgICBdCiAgICAgIH0sCiAgICAgICJyZXBvcnRzL2thZ2dsZV9ydW5zLzIwMjYxMDAzX2ZpeHR1cmVfaW50ZWdyaXR5L3J1bl9wcm92ZW5hbmNlLmpzb24iOiB7CiAgICAgICAgInNoYTI1NiI6ICI1NmE4MjE4ZjJkNmJiZTc1MDg4M2U0N2RkNmZmODM4YzM1Y2FmY2Q1OWVjYmMzMDZlYjI2NTIzOTNiNzhhZGI3IiwKICAgICAgICAicmV0YWluZWRfaWRzIjogW10KICAgICAgfSwKICAgICAgInJlcG9ydHMva2FnZ2xlX3J1bnMvMjAyNjEwMDNfc21va2UvYW5hbHlzaXMuanNvbiI6IHsKICAgICAgICAic2hhMjU2IjogIjNlMTA0ODhhNTM1MTUwNzBiYmEzMDA1NzJmZTc4Y2UwMTJkYzY3ZDlmNzBjYTNjZmY0ZWFiOTJjZWQ1MGEwN2MiLAogICAgICAgICJyZXRhaW5lZF9pZHMiOiBbXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9ldmFsX2NvbmZpZ19zb3VyY2VfcHJvYmUvcnVuX3Byb3ZlbmFuY2UuanNvbiI6IHsKICAgICAgICAic2hhMjU2IjogIjQ3ODk5MWVlNjA1ZWFhZmYxNzBmMzJlYjExZDNmZWZjMzQxMDFiN2Q5NTlmZGRhODk3YzBjNTRmMmMzOGMzM2UiLAogICAgICAgICJyZXRhaW5lZF9pZHMiOiBbXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9maXh0dXJlX2RlcGVuZGVuY2llcy9hbmFseXNpcy5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiOTM4YTU0MzE4MzBlMTRhYTA5Y2JjMWYxYjQyZTBlMjhmYzczMzQxMTFhNTQ5M2M2ZTRkZjA2MTM5MWUyODdiOSIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFsKICAgICAgICAgICJyZXF1ZXN0c182NTkyIiwKICAgICAgICAgICJyaWNoXzMwNjciCiAgICAgICAgXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9maXh0dXJlX2RlcGVuZGVuY2llcy9ydW5fcHJvdmVuYW5jZS5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiMjE0ZjBlZjcyNjk3MTUzNzU1NWEyMWI1YmFmNmY0MTgyOGMzODJjZDU4MTZkODllYTUyOWVmZmMxMTYzMDE1MyIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFtdCiAgICAgIH0sCiAgICAgICJyZXBvcnRzL2thZ2dsZV9ydW5zLzIwMjYxMDA0X2xlZ2FjeV9yZWRpcmVjdC9hbmFseXNpcy5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiOWIwMWYxOWNlZWE4Y2Q3ZjM3YTBmZWZlZjU4ZDFmNTAxNDkwNDY5ODQ2ZGI2MDM4Zjc0MTZlODI1ZTc4MTFmYiIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFsKICAgICAgICAgICJyZXF1ZXN0c182NTkyIiwKICAgICAgICAgICJyaWNoXzMwNjciCiAgICAgICAgXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9sZWdhY3lfcmVkaXJlY3QvcnVuX3Byb3ZlbmFuY2UuanNvbiI6IHsKICAgICAgICAic2hhMjU2IjogImU3NjcxMzBiMmE5NjgwMWY5ODRkNjRmYTQyOGVlOTUyZTI5ZDc3ZDJmNzZiNGU2NDJmMWJhOWQxMGFiOTc1MTMiLAogICAgICAgICJyZXRhaW5lZF9pZHMiOiBbXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9tb2JpbGVfc291cmNlX2F1ZGl0L3J1bl9wcm92ZW5hbmNlLmpzb24iOiB7CiAgICAgICAgInNoYTI1NiI6ICJmMmM1ZjAxZDQ5ZmQyMTM2NjYyNWE1NmYzNTkwYWM4OWU5ZTEwOWUxYWQzMWMxODZjZDQ0YmY0MDExMjNiYzRmIiwKICAgICAgICAicmV0YWluZWRfaWRzIjogW10KICAgICAgfSwKICAgICAgInJlcG9ydHMva2FnZ2xlX3J1bnMvMjAyNjEwMDRfbW9iaWxlX3NvdXJjZV9hdWRpdF9yZXRyeTEvcnVuX3Byb3ZlbmFuY2UuanNvbiI6IHsKICAgICAgICAic2hhMjU2IjogImI1OTQwYmZlOWM4MmQ0N2FkY2EwNzQ5NTkxNDA3OGMyZjY5MzY2ZjhkODg2NjY0NDYwYWUwYWY0MzFlMWQ5NGEiLAogICAgICAgICJyZXRhaW5lZF9pZHMiOiBbXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9tb2JpbGVfc291cmNlX2F1ZGl0X3JldHJ5Mi9ydW5fcHJvdmVuYW5jZS5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiODRjNjA1OGY5ZjVlZTdhZTQ4NmNlMjRiMmU2MzIxYjkzMmRiNzcwZThmMjJjZjI2NTcyY2Q4NmRkN2FkZTFhYSIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFtdCiAgICAgIH0sCiAgICAgICJyZXBvcnRzL2thZ2dsZV9ydW5zLzIwMjYxMDA0X3BhcGVyX2V4dHJhY3RlZF9yZXBsYXkvcnVuX3Byb3ZlbmFuY2UuanNvbiI6IHsKICAgICAgICAic2hhMjU2IjogImRkNGQ3Y2E5NjgyZWFmYjc0ZWNlOTY3NDgxM2RmZDIwMTAxZmMzYWE0ODBkZjRkYjBmMzM5Y2RlOWY2ZGVlYzEiLAogICAgICAgICJyZXRhaW5lZF9pZHMiOiBbXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9wdWJsaWNfZml4dHVyZV9jb250cm9scy9ydW5fcHJvdmVuYW5jZS5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiNzgwYjZhN2FmNDliNjJhODc1MWQ4NTY0NDc2MGMzNTdiNDRhYWFkNTY1OTNkN2RhMTc3OTE3NzhlMWYzOTgwMyIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFsKICAgICAgICAgICJmYXN0YXBpXzE0MDc3IiwKICAgICAgICAgICJmYXN0YXBpXzE0MjYyIiwKICAgICAgICAgICJyaWNoXzI3MjUiLAogICAgICAgICAgInJpY2hfNDA3OSIKICAgICAgICBdCiAgICAgIH0sCiAgICAgICJyZXBvcnRzL2thZ2dsZV9ydW5zLzIwMjYxMDA0X3JpY2hfZml4dHVyZV9xdWFsaWZpZWRfY2FwODE5Mi9hbmFseXNpcy5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiMWYwNzU2ZDliZDFkM2ZmZWUxYmQ3YmRlMzZlMmM5MjIzYTg3MzcxMzFlYTRkZmJkYWI3NGE0MzA5Njg2OWI4YyIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFsKICAgICAgICAgICJyaWNoXzI3MjUiLAogICAgICAgICAgInJpY2hfNDA3OSIKICAgICAgICBdCiAgICAgIH0sCiAgICAgICJyZXBvcnRzL2thZ2dsZV9ydW5zLzIwMjYxMDA0X3JpY2hfZml4dHVyZV9xdWFsaWZpZWRfY2FwODE5Mi9ydW5fcHJvdmVuYW5jZS5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiZjgyOTA5ZDQ0Nzk0NjUyZmIyMjdiMmQyYzJhN2M2NDI5Zjc2MGEwNTUzMTQwMzJmY2Q1YTRhMWU4OGQ0N2Q2NyIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFsKICAgICAgICAgICJyaWNoXzI3MjUiLAogICAgICAgICAgInJpY2hfNDA3OSIKICAgICAgICBdCiAgICAgIH0sCiAgICAgICJyZXBvcnRzL2thZ2dsZV9ydW5zLzIwMjYxMDA0X3NvdXJjZV9vbmx5X3YzL2FuYWx5c2lzLmpzb24iOiB7CiAgICAgICAgInNoYTI1NiI6ICJhN2YzMmFjODNjNDAxOWEwN2I4YjAwMjcyMTliYjg2ZTZiMTc3OGRiNTIxMjg1ZTZkNjg3NWUwYmNlMWEwZDRlIiwKICAgICAgICAicmV0YWluZWRfaWRzIjogW10KICAgICAgfSwKICAgICAgInJlcG9ydHMva2FnZ2xlX3J1bnMvMjAyNjEwMDRfc291cmNlX29ubHlfdjMvcnVuX3Byb3ZlbmFuY2UuanNvbiI6IHsKICAgICAgICAic2hhMjU2IjogImE3MGY0YTkyMDczMzA4ZmQ1NzAwYjU4ZGQzOTYyYTliYzk0YWY0ZmZhMmMzZjIwYWE5NDY0M2YxMzJlY2ZkYTkiLAogICAgICAgICJyZXRhaW5lZF9pZHMiOiBbXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9zb3VyY2Vfb25seV92M19iYXRjaC9ydW5fcHJvdmVuYW5jZS5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiMTUzYzc4ODQ5YzU5Zjg3NjhkODEzZDBlZmUxYWE2YjI4MjFiODhkYTY4ZmY0MGNkN2E5MjJlNmUzNjQ0YjQ2YyIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFtdCiAgICAgIH0sCiAgICAgICJyZXBvcnRzL2thZ2dsZV9ydW5zLzIwMjYxMDA0X3NvdXJjZV9vbmx5X3YzX2JhdGNoX3JldHJ5MS9hbmFseXNpcy5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiZmI4YjYwM2JmNWJjOWQ5NGZkMzdiMjZhNzhlODZlOWIyNWZjZjU4YzUzYTdhODg2ODliOTBhNDQ0YTU4YjJlNyIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFsKICAgICAgICAgICJyaWNoXzMwNjciCiAgICAgICAgXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNF9zb3VyY2Vfb25seV92M19iYXRjaF9yZXRyeTEvcnVuX3Byb3ZlbmFuY2UuanNvbiI6IHsKICAgICAgICAic2hhMjU2IjogIjE3NmI5ZjczYTgzYTQyMzY3NzYxMDhjODg4YjczMzNjZDc2OTVlM2MwNDA1NjYyMmMzZTk4ODJmMzg4YjM5OWUiLAogICAgICAgICJyZXRhaW5lZF9pZHMiOiBbXQogICAgICB9LAogICAgICAicmVwb3J0cy9rYWdnbGVfcnVucy8yMDI2MTAwNV9zb3VyY2VfYnVkZ2V0X2NvbXBhY3Rpb241X3RyYW5zZmVyX3JldmlzaW9uMS9ydW5fcHJvdmVuYW5jZS5qc29uIjogewogICAgICAgICJzaGEyNTYiOiAiY2MzMjcxNmJhNjA5MDBkN2NlMjlhOWExZmFlNDBiYmJhNjNjNTEwNmE4OGQ2NWEyOTIyZTFmZmRjYmZjYWFmYSIsCiAgICAgICAgInJldGFpbmVkX2lkcyI6IFsKICAgICAgICAgICJyaWNoXzI3MjUiLAogICAgICAgICAgInJpY2hfNDA3OSIKICAgICAgICBdCiAgICAgIH0KICAgIH0sCiAgICAic291cmNlX3Byb2JsZW1fb25seV92aWV3cyI6IHsKICAgICAgImJlbmNobWFya3MvY29tcGV0aXRpb24vc3R1ZGllcy8yMDI2MTAwM192MS9mb2xkXzAwL2FnZW50L2NhbGlicmF0aW9uLmpzb25sIjogImE5Mzc2NGM5OGNjYzY1ODUxOTkyODIwYzVlN2Y1Y2FjMTE3MGQ1NGVkMDljYzJkMzM1NTA5OGE1OWQzMzBhNTYiLAogICAgICAiYmVuY2htYXJrcy9jb21wZXRpdGlvbi9zdHVkaWVzLzIwMjYxMDAzX3YxL2ZvbGRfMDAvYWdlbnQvZml0Lmpzb25sIjogIjQ4OTlhYTg0OTEwODQ3NzE5ZWY2ZjI0MjRiMjk1ZTU3MWJhMGU3OGRjMWViNDA2NzIwYzQ3MzVkMDZmMzUyYzQiLAogICAgICAiYmVuY2htYXJrcy9jb21wZXRpdGlvbi9zdHVkaWVzLzIwMjYxMDAzX3YxL2ZvbGRfMDAvYWdlbnQvaGVsZF9vdXQuanNvbmwiOiAiY2JhYzY5Y2U3YzZiOGQ2ODdjYmNiODFhYjEyNmNjNDY0MjcwMzRhNmY1ZDU0Y2FiNWYxYWM5NTM5N2U4MDM5ZiIsCiAgICAgICJiZW5jaG1hcmtzL2NvbXBldGl0aW9uL3N0dWRpZXMvMjAyNjEwMDNfdjEvZm9sZF8wMS9hZ2VudC9jYWxpYnJhdGlvbi5qc29ubCI6ICIxMmIxYzBjNTY5OWU0MDk4MGYwOGJhM2FkZDNjMTE0NTU0NTRmNzQzNDYxZGY4NzEwYTZjN2ZmNjBlNzk2NjJiIiwKICAgICAgImJlbmNobWFya3MvY29tcGV0aXRpb24vc3R1ZGllcy8yMDI2MTAwM192MS9mb2xkXzAxL2FnZW50L2ZpdC5qc29ubCI6ICJhMjgyNzk4NTM2MWQzNWZjMDlmMGMyMjAxZTZlN2M2NDdkNDQ2OWVlMWVlOWFmYjQxODVjMTZkMTEzNGIyOTU0IiwKICAgICAgImJlbmNobWFya3MvY29tcGV0aXRpb24vc3R1ZGllcy8yMDI2MTAwM192MS9mb2xkXzAxL2FnZW50L2hlbGRfb3V0Lmpzb25sIjogImE5Mzc2NGM5OGNjYzY1ODUxOTkyODIwYzVlN2Y1Y2FjMTE3MGQ1NGVkMDljYzJkMzM1NTA5OGE1OWQzMzBhNTYiLAogICAgICAiYmVuY2htYXJrcy9jb21wZXRpdGlvbi9zdHVkaWVzLzIwMjYxMDAzX3YxL2ZvbGRfMDIvYWdlbnQvY2FsaWJyYXRpb24uanNvbmwiOiAiNTQwMjU4OWQxYTA0YWFmOGUwNDIwZDBkMTRkOWFhYWQ4MmY4YTIyMjY4NjJmNTMzZTdjYmYyMDQyYjM5NzY2ZiIsCiAgICAgICJiZW5jaG1hcmtzL2NvbXBldGl0aW9uL3N0dWRpZXMvMjAyNjEwMDNfdjEvZm9sZF8wMi9hZ2VudC9maXQuanNvbmwiOiAiNjQzODFjYjg5NDY3MWQ3NWRlYjcwNDBlMGFiOWMwY2Y5ODMwZjQ4YzBjNGRlMDAyMjI5ZmM3MjRmNmFkNzIwMyIsCiAgICAgICJiZW5jaG1hcmtzL2NvbXBldGl0aW9uL3N0dWRpZXMvMjAyNjEwMDNfdjEvZm9sZF8wMi9hZ2VudC9oZWxkX291dC5qc29ubCI6ICIxMmIxYzBjNTY5OWU0MDk4MGYwOGJhM2FkZDNjMTE0NTU0NTRmNzQzNDYxZGY4NzEwYTZjN2ZmNjBlNzk2NjJiIiwKICAgICAgImJlbmNobWFya3MvY29tcGV0aXRpb24vc3R1ZGllcy8yMDI2MTAwM192MS9mb2xkXzAzL2FnZW50L2NhbGlicmF0aW9uLmpzb25sIjogImNiYWM2OWNlN2M2YjhkNjg3Y2JjYjgxYWIxMjZjYzQ2NDI3MDM0YTZmNWQ1NGNhYjVmMWFjOTUzOTdlODAzOWYiLAogICAgICAiYmVuY2htYXJrcy9jb21wZXRpdGlvbi9zdHVkaWVzLzIwMjYxMDAzX3YxL2ZvbGRfMDMvYWdlbnQvZml0Lmpzb25sIjogImQ2ODRjMmM3OWQ0MDRkMmQ2MzQwYzNkOTYwM2MwMDg2MTgwMjdkM2M4MjhhZDdiNjMzMGYwNmRmZmE5NDIyNTYiLAogICAgICAiYmVuY2htYXJrcy9jb21wZXRpdGlvbi9zdHVkaWVzLzIwMjYxMDAzX3YxL2ZvbGRfMDMvYWdlbnQvaGVsZF9vdXQuanNvbmwiOiAiNTQwMjU4OWQxYTA0YWFmOGUwNDIwZDBkMTRkOWFhYWQ4MmY4YTIyMjY4NjJmNTMzZTdjYmYyMDQyYjM5NzY2ZiIKICAgIH0sCiAgICAiaXNzdWVfdGV4dF9vcl9yZWZlcmVuY2Vfb3Jfb3V0Y29tZV91c2VkX3RvX3JhbmsiOiBmYWxzZSwKICAgICJyZWZlcmVuY2Vfdmlld3NfcmVhZF9ieV9wcmVwYXJhdGlvbiI6IGZhbHNlCiAgfSwKICAia25vd25fZGV2ZWxvcG1lbnRfcmVzdWx0cyI6IHsKICAgICJwYXJlbnQxOF9hY3R1YWxfYW5hbHlzaXNfc2hhMjU2IjogIjFmMDc1NmQ5YmQxZDNmZmVlMWJkN2JkZTM2ZTJjOTIyM2E4NzM3MTMxZWE0ZGZiZGFiNzRhNDMwOTY4NjliOGMiLAogICAgInBhcmVudDE4X3Jlc3VsdCI6ICJzb3VyY2UxLzJfcGF0aDEvMl9pZGVudGljYWxfc3VjY2Vzc2Z1bF9wYXRjaF9ub19nYWluIiwKICAgICJ0cmFuc2ZlcjIxX3Byb3ZlbmFuY2Vfc2hhMjU2IjogImNjMzI3MTZiYTYwOTAwZDdjZTI5YTlhMWZhZTQwYmJiYTYzYzUxMDZhODhkNjVhMjkyMmUxZmZkY2JmY2FhZmEiLAogICAgInRyYW5zZmVyMjFfbGFzdF9yZXRhaW5lZF9zdGF0ZSI6ICJxdWV1ZWQiLAogICAgInRoZXNlX2ZvdXJfY2FzZV9tb2RlbF9vdXRjb21lc19hdmFpbGFibGVfYXRfZnJlZXplIjogZmFsc2UKICB9LAogICJsaW1pdHMiOiB7CiAgICAidGltZW91dF9zZWNvbmRzIjogMzAwLAogICAgIm1heF90aW1lX21pbnV0ZXMiOiAyMCwKICAgICJtYXhfdG9vbF9jYWxscyI6IDUwLAogICAgIm1heF90dXJucyI6IDEwMAogIH0sCiAgInRpbWVvdXRfc2VtYW50aWNzIjogIlBlciBzYW5kYm94IGNvbW1hbmQgaW5jbHVkaW5nIHZlcmlmaWVyIHB5dGVzdDsgbm8gd2hvbGUtdmVyaWZpZXIgZGVhZGxpbmUiLAogICJmaXh0dXJlX2NvbnRyb2xzIjogewogICAgImFjY2VsZXJhdG9yIjogIm5vbmUiLAogICAgIm1vZGVsX2luZmVyZW5jZSI6IGZhbHNlLAogICAgImFzc2lnbmVkX2NoZWNrcyI6IDgsCiAgICAiY29uZGl0aW9ucyI6IFsKICAgICAgImNsZWFuX2Jhc2UiLAogICAgICAib2ZmaWNpYWxfcmVmZXJlbmNlX3BhdGNoIgogICAgXSwKICAgICJyZWZlcmVuY2VfYWNjZXNzIjogIkluZGVwZW5kZW50IENQVSB2ZXJpZmllciBvbmx5OyBmdWxsIHJlZmVyZW5jZSBwYXRjaGVzIHN0YXkgaW4gbWVtb3J5LiBQcml2YXRlIHRlc3Qgb3V0cHV0IGFuZCBncmFkZXIgZXJyb3IgdGV4dCBhcmUgcmV0YWluZWQgc29sZWx5IGZvciBlbnZpcm9ubWVudCBkaWFnbm9zaXMuIgogIH0sCiAgImFybXMiOiB7fSwKICAic2NoZWR1bGUiOiBbXSwKICAiZnV0dXJlX21vZGVsX2dhdGUiOiAiUmVjb25jaWxlMjEgYmVmb3JlIG5ldyBHUFUgc3BlbmRpbmc7IGEgc2VwYXJhdGUgZnJvemVuIHBsYW4gbXVzdCBiaW5kIGFueSBhcm1zIGFuZCBtb2RlbCBhc3NpZ25tZW50cyIsCiAgImZ1dHVyZV9jb250cm9sX3NlbGVjdGlvbl9ydWxlIjogIkNob29zZSBmaXJzdCBtZXRhZGF0YS1yYW5rZWQgY2xlYW5fYmFzZV9ub25wYXNzaW5nX3JlZmVyZW5jZV92ZXJpZmllZCBjYXNlIHBlciByZXBvc2l0b3J5IGZvciBhIHByb3NwZWN0aXZlIGZvdXItc2xvdCBwYWlyZWQgZGV2ZWxvcG1lbnQgZGlhZ25vc3RpYzsgaWYgYW55IHJlcG9zaXRvcnkgaGFzIG5vIHF1YWxpZmllZCBjYXNlLCByZXBvcnQgaW5zdWZmaWNpZW50IGNvdmVyYWdlIHdpdGhvdXQgcmVwbGFjZW1lbnQiLAogICJzdG9wcGluZ19ydWxlIjogIlNhbWUgZWlnaHQgY2hlY2tzIGFuZCBvcmRlcjsgc2F2ZSBhY3R1YWwgZmFpbHVyZSBhbmQgaGFsdCBvbiByYWlzZWQgaW5mcmFzdHJ1Y3R1cmUgZXhjZXB0aW9uLiBObyByZWZpbGwsIHJldHJ5LCBtb2RlbCBpbmZlcmVuY2Ugb3IgZW52aXJvbm1lbnQgY2hhbmdlcy4iLAogICJyZXNlYXJjaF9hc3Npc3RhbnRfcHVibGljX3Bvb2xfcmVmZXJlbmNlX2V4cG9zdXJlX3JlY29yZGVkIjogdHJ1ZSwKICAibmV3X3Byb2JsZW1fdGV4dF9wcmVwYXJhdGlvbl9leHBvc3VyZSI6ICJPbmx5IGV4aXN0aW5nIGZpdmUtZmllbGQgdmlld3M7IGVhcmxpZXIgbWV0YWRhdGEgaW5zcGVjdGlvbiBwcmludGVkIHNldmVyYWwgUmVxdWVzdHMgcHJvYmxlbSBzdGF0ZW1lbnRzLCBpbmNsdWRpbmc3NDMzOyBubyBldmFsdWF0b3IgcmVmZXJlbmNlIHRleHQgcmVhZCBoZXJlIiwKICAiaG9sZG91dF9vcl9jb25maXJtYXRvcnlfY2xhaW1fYWxsb3dlZCI6IGZhbHNlLAogICJhZ2VudF9nZW5lcmF0ZWRfcmVwYWlycyI6IG51bGwsCiAgIm9mZmljaWFsX2xlYWRlcmJvYXJkX3Njb3JlIjogbnVsbCwKICAiY29tcGV0aXRpb25fZW50cnlfc3VibWl0dGVkIjogZmFsc2UsCiAgIlVTQl90ZXN0aW5nIjogImRlZmVycmVkX2F0X3VzZXJfcmVxdWVzdCIsCiAgInBhcmVudDIyX3BsYW5fc2hhMjU2IjogImE5YmFhZDJlZmFhYzZhODlkMGJjNDE2ZTcwM2ZhYjFlMTczMjUwNDg4ZTJhYmY4NDgxMjNkMzA0YTA1OTI0ZjUiLAogICJwYXJlbnQyMl9hbmFseXNpc19zaGEyNTYiOiAiZTRlNTU4NzBjMDAzOTU4YjQzMjNiM2QzMTU1NDZhNmVhYTE4ZjRmMWZhYmM2YzhmZDgyZGI0MTFiZmQyZjI5NSIsCiAgInBhcmVudDIyX2tub3duX2NsYXNzaWZpY2F0aW9ucyI6IFsKICAgIHsKICAgICAgInRhc2tfaWQiOiAicmVxdWVzdHNfNjY0NCIsCiAgICAgICJjbGFzc2lmaWNhdGlvbiI6ICJjbGVhbl9iYXNlX2FscmVhZHlfcGFzc2VzIgogICAgfSwKICAgIHsKICAgICAgInRhc2tfaWQiOiAiZmFzdGFwaV8xNTAyMyIsCiAgICAgICJjbGFzc2lmaWNhdGlvbiI6ICJvZmZpY2lhbF9yZWZlcmVuY2Vfbm90X3ZlcmlmaWVkIgogICAgfSwKICAgIHsKICAgICAgInRhc2tfaWQiOiAicmVxdWVzdHNfNzQzMyIsCiAgICAgICJjbGFzc2lmaWNhdGlvbiI6ICJvZmZpY2lhbF9yZWZlcmVuY2Vfbm90X3ZlcmlmaWVkIgogICAgfSwKICAgIHsKICAgICAgInRhc2tfaWQiOiAiZmFzdGFwaV8xNDUxMiIsCiAgICAgICJjbGFzc2lmaWNhdGlvbiI6ICJvZmZpY2lhbF9yZWZlcmVuY2Vfbm90X3ZlcmlmaWVkIgogICAgfQogIF0sCiAgInRyYW5zZmVyMjFfY29tcGxldGVkX3Byb3ZlbmFuY2Vfc2hhMjU2IjogImZkYTZjYTYxMTcwNGQzYjZmOGU4NGE3ZTNmZmYwZGY5ZGY1Mzg0ZjdkNDdkM2QwNjY5MzhkYmZlYTQ2NTE4NGEiLAogICJ0cmFuc2ZlcjIxX2tub3duX3Jlc3VsdCI6ICIxX29mXzJfYm90aF81MF90b29sc19ub19nYWluX29yX3NlbGVjdGlvbiIsCiAgIm5ld19jYXNlc19zZWxlY3RlZCI6IGZhbHNlLAogICJlbnZpcm9ubWVudF9tb2RpZmljYXRpb25fZnJvbV9wYXJlbnQyMiI6IGZhbHNlLAogICJwcml2YXRlX2dyYWRlcl90ZXh0X3JldGFpbmVkIjogdHJ1ZSwKICAicmVmZXJlbmNlX3BhdGNoX2V4cG9ydGVkIjogZmFsc2UsCiAgImFnZW50X29yX3B1YmxpY19yZXNvdXJjZV9pbnB1dCI6IGZhbHNlLAogICJkaWFnbm9zdGljX2ZvbGxvd3VwX3J1bGUiOiAiUmV2aWV3IGFjdHVhbCBncmFkZXIgZXJyb3JzIGJlZm9yZSBkZWNsYXJpbmcgYSBzZXBhcmF0ZSBlbnZpcm9ubWVudCByZXBhaXI7IGRvIG5vdCBtdXRhdGUgdGhlc2UgZnJvemVuIGNvbnRyb2xzIG9yIHRyZWF0IHRoZWlyIHJlZmVyZW5jZSBvdXRjb21lcyBhcyBtb2RlbCByZXBhaXJzIgp9Cg=='
PACKAGES = json.loads('{}')
SOURCE_BINDING = json.loads('{"plan_sha256": "78b02914e4d5359735b3ca0bd3834b8900b9e5a413dce77f4222afe4dbb9e505", "problem_only_input_sha256": "4dbfcbb2151487854269975eaed936370d7ac9a003b48523a2b2362430fc23f4", "source_sha256": {"scripts/prepare_crossrepo_fixture_diagnostic.py": "b81995ce15613e63ea3d4d3447fb78278397034c4ccd8e1037fc5c6314485d34", "scripts/build_tool_path_comparison_notebooks.py": "6df902c98f4a9238cbfec83a8a6d94843b82abe2d0865b6c681375be0ba14115", "competition/private_fixture_diagnostics.py": "e9d82e697a31f4c2b4c122e2787d9712f8f08d20fca3a5f4f963148291b6dc87", "competition/public_fixture_controls.py": "6a1fe062faf69188e584bcbd8ffa6147e64b6bb65bd2588f5c2fb46beedd4af8", "competition/smoke_artifacts.py": "b0cc8414c2f76d61af3bf4e9d7f0720b99a762ac428611cf86b23b80c384b2a3", "competition/mobile_audit_environment.py": "586c215814c09dd0bbbe56a888e20032a5cbd262b4b1491ae7c05c42552659f1", "competition/mobile_source_audit_retry.py": "afd8f56666eebe7bfad7a48270ef7c3451a37c21760ef370887de1d4e8c930e1", "competition/declared_comparison_environment.py": "0e3b118f3ff82a8243e28729a20df0019027c1def6b039a2df2e43b8ab9d9eb9", "competition/development_comparison.py": "1b2e9de9844b0017a178032ae14c11dc6712099777d4c5013ab37a9f63228c63", "competition/fixture_integrity.py": "9a59465a39b3ad949b05896d64d37f73c5c66d25e60ab9f9b3ab746b99ffdfa6", "kaggle/22_crossrepo_public_fixture_controls.ipynb": "3dbf6359cb937249ebcf6e37869745b129712bddcfefb8479b44371ee385df25"}, "selected_case_ids": ["requests_6644", "fastapi_15023", "requests_7433", "fastapi_14512"], "no_agent_package_needed_for_CPU_controls": true, "private_diagnostic_exports_never_agent_or_public_resource_input": true}')
AGENT_INPUT_PACKET_BASE64 = 'eyJiYXNlX2NvbW1pdCI6ICIzODJmYzJjMGM2YzBlZjA4NzRiYzY1YmMxMTc1Zjk3YzA3M2U1MDg2IiwgImhpbnRzX3RleHQiOiAiIiwgImluc3RhbmNlX2lkIjogInJlcXVlc3RzXzY2NDQiLCAicHJvYmxlbV9zdGF0ZW1lbnQiOiAiVHJpbSBleGNlc3MgbGVhZGluZyBwYXRoIHNlcGFyYXRvcnNcblxuQSBVUkwgd2l0aCBleGNlc3MgbGVhZGluZyAvIChwYXRoLXNlcGFyYXRvcilzIHdvdWxkIGNhdXNlIHVybGxpYjMgdG8gYXR0ZW1wdCB0byByZXBhcnNlIHRoZSByZXF1ZXN0LXVyaSBhcyBhIGZ1bGwgVVJJIHdpdGggYSBob3N0IGFuZCBwb3J0LiBUaGlzIGJ5cGFzc2VzIHRoYXQgbG9naWMgaW4gQ29ubmVjdGlvblBvb2wudXJsb3BlbiBieSByZXBsYWNpbmcgdGhlc2UgbGVhZGluZyAvcyB3aXRoIGp1c3QgYSBzaW5nbGUgLy5cclxuXHJcbkNsb3NlcyAjNjY0MyIsICJyZXBvIjogInBzZi9yZXF1ZXN0cyJ9CnsiYmFzZV9jb21taXQiOiAiYzA4MzZkYzFiN2Y0YmE5MWU5NTYzYzhjNGNjZDBlYmU2NTViYzkxNCIsICJoaW50c190ZXh0IjogIiIsICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE1MDIzIiwgInByb2JsZW1fc3RhdGVtZW50IjogIvCfk50gVXBkYXRlIGRvY3MgZm9yIHJlc3BvbnNlcyBhbmQgbmV3IHN0cmVhbSB3aXRoIGB5aWVsZGBcblxu8J+TnSBVcGRhdGUgZG9jcyBmb3IgcmVzcG9uc2VzIGFuZCBuZXcgc3RyZWFtIHdpdGggYHlpZWxkYCIsICJyZXBvIjogImZhc3RhcGkvZmFzdGFwaSJ9CnsiYmFzZV9jb21taXQiOiAiMGI0MDFjNzZiNmU4MGE0ZWVjZjNjNjkwMDg1YjI1NTNmNmUyNjFjYSIsICJoaW50c190ZXh0IjogIiIsICJpbnN0YW5jZV9pZCI6ICJyZXF1ZXN0c183NDMzIiwgInByb2JsZW1fc3RhdGVtZW50IjogIkZpeCBgcHJlcGFyZV9ib2R5YCBzdHJlYW0gZGV0ZWN0aW9uIGZvciBgX19nZXRhdHRyX19gLWJhc2VkIGZpbGUgd3JhcHBlcnNcblxuUmVzb2x2ZXMgIzc0MzIgLlxyXG5cclxuQWRkIGBoYXNhdHRyKGRhdGEsIFwiX19pdGVyX19cIilgIGFzIGEgZmFsbGJhY2sgc3RyZWFtIGRldGVjdG9yOlxyXG5gYGBweXRob25cclxuaWYgKFxyXG4gICAgaXNpbnN0YW5jZShkYXRhLCBJdGVyYWJsZSkgb3IgaGFzYXR0cihkYXRhLCBcIl9faXRlcl9fXCIpXHJcbikgYW5kIG5vdCBpc2luc3RhbmNlKGRhdGEsIChzdHIsIGJ5dGVzLCBsaXN0LCB0dXBsZSwgTWFwcGluZykpOlxyXG5gYGBcclxuVGhpcyBjYXRjaGVzIGZpbGUtbGlrZSBvYmplY3RzIHRoYXQgcHJveHkgdGhlaXIgaW50ZXJmYWNlIHRocm91Z2ggYF9fZ2V0YXR0cl9fYCwgcmVzdG9yaW5nIHRoZSAyLjMzLjEgYmVoYXZpb3Igd2l0aG91dCByZXZlcnRpbmcgdGhlIGlzaW5zdGFuY2UoSXRlcmFibGUpIG1vZGVybml6YXRpb24gZm9yIHN0YW5kYXJkIGl0ZXJhYmxlcy4iLCAicmVwbyI6ICJwc2YvcmVxdWVzdHMifQp7ImJhc2VfY29tbWl0IjogIjFmY2VjODhhZDJkZjQ1NDFiNmE4MWM0Nzg2YzRkMDM5ZGM3NDVlMWUiLCAiaGludHNfdGV4dCI6ICIiLCAiaW5zdGFuY2VfaWQiOiAiZmFzdGFwaV8xNDUxMiIsICJwcm9ibGVtX3N0YXRlbWVudCI6ICLwn5CbIEZpeCBzdXBwb3J0IGZvciB0YWdnZWQgdW5pb24gd2l0aCBkaXNjcmltaW5hdG9yIGluc2lkZSBvZiBgQW5ub3RhdGVkYCB3aXRoIGBCb2R5KClgXG5cbvCfkJsgRml4IHN1cHBvcnQgZm9yIHRhZ2dlZCB1bmlvbiB3aXRoIGRpc2NyaW1pbmF0b3IgaW5zaWRlIG9mIGBBbm5vdGF0ZWRgIHdpdGggYEJvZHkoKWBcclxuXHJcblJlbGF0ZWQgdG86IGh0dHBzOi8vZ2l0aHViLmNvbS9mYXN0YXBpL2Zhc3RhcGkvZGlzY3Vzc2lvbnMvMTQ0OTUsIGh0dHBzOi8vZ2l0aHViLmNvbS9mYXN0YXBpL2Zhc3RhcGkvaXNzdWVzLzE0NTA4IiwgInJlcG8iOiAiZmFzdGFwaS9mYXN0YXBpIn0K'
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    RUN = Path('/kaggle/working') / ('private_diagnostic23_' + MODE + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    frozen_plan_bytes = base64.b64decode(PLAN_BYTES_BASE64, validate=True)
    if hashlib.sha256(frozen_plan_bytes).hexdigest() != SOURCE_BINDING['plan_sha256'] or json.loads(frozen_plan_bytes) != PLAN:
        raise ValueError('Frozen plan bytes differ from source binding')
    (RUN / 'plan.json').write_bytes(frozen_plan_bytes)
    (RUN / 'source_binding.json').write_text(json.dumps(SOURCE_BINDING, indent=2) + '\n', encoding='utf-8')
    packet = base64.b64decode(AGENT_INPUT_PACKET_BASE64, validate=True)
    if hashlib.sha256(packet).hexdigest() != PLAN['selected_problem_only_input_sha256']:
        raise ValueError('Problem-only input differs from the frozen plan')
    (RUN / 'agent_inputs.jsonl').write_bytes(packet)
    for arm, package in PACKAGES.items():
        payload = base64.b64decode(package['base64'], validate=True)
        if hashlib.sha256(payload).hexdigest() != package['sha256']:
            raise ValueError('Candidate package differs from frozen bytes')
        destination = RUN / 'agents' / arm
        destination.mkdir(parents=True)
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            if set(archive.namelist()) != {'agent.yaml', 'prompts/system.md', 'configs/sampling.yaml'}:
                raise ValueError('Unexpected candidate package members')
            archive.extractall(destination)
        (RUN / (arm + '.zip')).write_bytes(payload)
    export_run(RUN)
    print('Frozen output directory:', RUN)


In [ ]:
"""Fail-fast, recorded CPU setup for the unchanged organizer source audit.

Definitions do not install packages. Runtime checks never filter incompatible
wheels into a superficially successful setup or establish scoring equivalence.
"""
from datetime import datetime, timezone
import glob
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import sysconfig
import traceback


REQUIRED_CPU_RUNTIME = {
    "implementation": "CPython", "python_major_minor": [3, 12],
    "system": "Linux", "machine": "x86_64",
}


def environment_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def environment_package_versions(names):
    values = {}
    for name in sorted(names):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def cpu_runtime_snapshot(package_names):
    return {
        "implementation": platform.python_implementation(),
        "python_major_minor": list(sys.version_info[:2]),
        "python_version": sys.version, "executable": sys.executable,
        "executable_realpath": str(Path(sys.executable).resolve()),
        "system": platform.system(), "machine": platform.machine(),
        "platform": platform.platform(), "libc": list(platform.libc_ver()),
        "soabi": sysconfig.get_config_var("SOABI"),
        "multiarch": sysconfig.get_config_var("MULTIARCH"),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "package_versions": environment_package_versions(package_names),
    }


def cpu_runtime_errors(runtime):
    return [key + " must be " + repr(expected) + "; observed " + repr(runtime.get(key))
            for key, expected in REQUIRED_CPU_RUNTIME.items()
            if runtime.get(key) != expected]


def normalized_organizer_wheel_name(name):
    # Retain the organizer workaround already used in frozen notebook 10.
    return name.replace("cu128", "+cu128") if "cu128" in name and "+" not in name else name


def classify_organizer_wheels(paths, supported_tags=None):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags() if supported_tags is None else supported_tags)
    rows = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.is_file():
            raise ValueError("Missing organizer wheel: " + str(path))
        row = {"source_path": str(path), "original_name": path.name,
               "install_name": normalized_organizer_wheel_name(path.name),
               "bytes": path.stat().st_size, "sha256": environment_file_sha256(path)}
        if "cutlass" in path.name.lower():
            row.update(selected=False, compatible=None,
                       exclusion_reason="existing_organizer_cutlass_workaround")
        else:
            try:
                distribution, version, build, tags = parse_wheel_filename(row["install_name"])
                row.update(selected=True, compatible=bool(tags & supported),
                           distribution=distribution, version=str(version),
                           wheel_tags=sorted(map(str, tags)),
                           matching_runtime_tags=sorted(map(str, tags & supported)))
            except ValueError as error:
                row.update(selected=True, compatible=False, wheel_parse_error=str(error))
        rows.append(row)
    if len({row["install_name"] for row in rows}) != len(rows):
        raise ValueError("Ambiguous organizer wheel aliases")
    return rows


def remove_recorded_cutlass_hooks():
    removed = []
    for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
                    "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for name in sorted(glob.glob(pattern)):
            path = Path(name)
            row = {"path": str(path), "sha256": environment_file_sha256(path)}
            path.unlink()
            removed.append(row)
    return removed


def source_audit_import_check():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    count = torch.cuda.device_count()
    if count != 0:
        raise ValueError("Select Accelerator None: source audit requires zero CUDA devices")
    return {"torch_version": torch.__version__, "cuda_device_count": count,
            "required_organizer_imports_succeeded": True}


def prepare_source_audit_environment(wheelhouse_dir, run_root, expected_packages,
                                     checkpoint_exporter, source_binding):
    """Install the whole selected organizer wheelhouse only after compatibility.

    Always retain runtime identity and setup failures before allowing the exact
    source inventory. UI labels are recorded separately from verified runtime.
    """
    run_root = Path(run_root)
    wheelhouse_dir = Path(wheelhouse_dir)
    run_root.mkdir(parents=True, exist_ok=False)
    package_names = set(expected_packages) | {"torch", "vllm", "pip", "packaging"}
    report = {
        "scope": "CPU_source_audit_environment_setup_not_model_or_grading",
        "created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "required_runtime": REQUIRED_CPU_RUNTIME, "source_binding": source_binding,
        "before": cpu_runtime_snapshot(package_names), "after": None,
        "wheelhouse_path": str(wheelhouse_dir), "wheel_manifest": [],
        "selected_wheels_omitted_for_incompatibility": [],
        "cutlass_hooks_removed": [], "pip_command": None, "pip_return_code": None,
        "model_inference_executed": False, "source_inventory_executed": False,
        "tests_executed": False, "private_scoring_equivalence_established": False,
        "runtime_image_id_verified": False,
        "requested_UI_profile": "original 2026-07-01 image, Accelerator None, Internet off",
    }

    def save():
        (run_root / "environment_receipt.json").write_text(
            json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("MOBILE_SOURCE_AUDIT_ENVIRONMENT=" + json.dumps({
            "status": report["status"], "receipt": str(run_root / "environment_receipt.json"),
            "python_version": report["before"]["python_version"],
            "runtime_errors": report.get("runtime_errors", []),
            "incompatible_wheels": report.get("incompatible_wheels", []),
            "pip_return_code": report["pip_return_code"],
        }), flush=True)
        checkpoint_exporter(run_root)

    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise RuntimeError("Attach the official organizer wheelhouse before setup")
        report["wheel_manifest"] = classify_organizer_wheels(wheelhouse_dir.glob("*.whl"))
        selected = [row for row in report["wheel_manifest"] if row["selected"]]
        report["runtime_errors"] = cpu_runtime_errors(report["before"])
        report["incompatible_wheels"] = [row["original_name"] for row in selected if not row["compatible"]]
        if not selected or report["runtime_errors"] or report["incompatible_wheels"]:
            report["status"] = "preflight_rejected_before_install"
            raise RuntimeError("CPU source-audit preflight rejected this environment; "
                               "use a verified Linux x86_64 CPython 3.12 worker. "
                               "Incompatible organizer wheels were retained and no install ran.")
        report["status"] = "preflight_passed"
        save()
        links = run_root / "organizer_wheel_links"
        links.mkdir()
        for row in selected:
            (links / row["install_name"]).symlink_to(Path(row["source_path"]).resolve())
        report["cutlass_hooks_removed"] = remove_recorded_cutlass_hooks()
        command = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                   "--no-deps", "--force-reinstall"] + [str(links / row["install_name"]) for row in selected]
        report.update(status="installing", pip_command=command)
        save()
        result = subprocess.run(command, capture_output=True, text=True)
        (run_root / "organizer_install_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
        (run_root / "organizer_install_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
        report["pip_return_code"] = result.returncode
        importlib.invalidate_caches()
        report["after"] = cpu_runtime_snapshot(package_names)
        if result.returncode != 0:
            report["status"] = "organizer_install_failed"
            raise RuntimeError("Organizer wheel installation failed; inspect retained install logs")
        observed = {name: report["after"]["package_versions"][name] for name in expected_packages}
        if observed != expected_packages:
            report["status"] = "organizer_core_identity_rejected"
            raise ValueError("Installed organizer core packages differ from the frozen source-audit plan")
        report["status"] = "checking_CPU_organizer_imports"
        save()
        report.update(source_audit_import_check())
        report["status"] = "ready_for_unchanged_source_audit"
        save()
        return report
    except Exception as error:
        if report["status"] in {"started", "preflight_passed", "installing", "checking_CPU_organizer_imports"}:
            report["status"] = "setup_exception_before_source_audit"
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / "setup_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
def resolve_organizer_audit_api(evaluator_module):
    verifier = getattr(evaluator_module, "verify_task", None)
    if not callable(verifier):
        raise ValueError("Installed Evaluator module has no callable official verify_task")
    implementation = inspect.getmodule(verifier)
    namespace = getattr(verifier, "__globals__", {})
    apis = {"verify_task": verifier}
    for name in ("resolve_task_snapshot_paths", "extract_snapshot", "sandbox_start", "sandbox_exec", "sandbox_stop"):
        # verify_task's own globals are the callables used by its retained source.
        # Snapshot resolution can be a public reexport from Evaluator's module.
        function = namespace.get(name)
        if function is None and implementation is not None:
            function = getattr(implementation, name, None)
        if function is None and name == "resolve_task_snapshot_paths":
            function = getattr(evaluator_module, name, None)
        if not callable(function):
            raise ValueError("Official verifier implementation lacks callable " + name)
        apis[name] = function
    placeholder = object()
    calls = {
        "resolve_task_snapshot_paths": ((placeholder, "instance_id", "repo"), {}),
        "extract_snapshot": ((placeholder, placeholder, placeholder),
                             {"patch_path": placeholder, "base_snapshot_path": placeholder}),
        "sandbox_start": ((placeholder,), {}),
        "sandbox_exec": ((placeholder, placeholder, "command"), {}),
        "sandbox_stop": ((placeholder, placeholder), {}),
    }
    for name, (args, kwargs) in calls.items():
        inspect.signature(apis[name]).bind(*args, **kwargs)
    for name in ("sandbox_start", "sandbox_exec", "sandbox_stop"):
        if not inspect.iscoroutinefunction(apis[name]):
            raise ValueError("Official sandbox callable is not asynchronous: " + name)
    return apis


In [ ]:
"""Recorded CPU/GPU preflight for separately frozen future public comparisons."""
from datetime import datetime, timezone
import hashlib
import importlib
import inspect
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import traceback



def declared_backend_check(mode, torch):
    count = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(index) for index in range(count)]
    if mode == 'cpu_controls':
        if count != 0:
            raise ValueError('Select Accelerator None for fixture controls')
    elif mode == 'gpu_comparison':
        if count != 4 or any('L4' not in name for name in names):
            raise ValueError('The declared comparison requires exactly four NVIDIA L4 GPUs')
    else:
        raise ValueError('Unknown declared experiment mode')
    return {'cuda_device_count': count, 'gpu_names': names, 'torch_version': torch.__version__}


def prepare_declared_environment(wheelhouse_dir, run_root, plan, checkpoint_exporter, mode):
    run_root, wheelhouse_dir = Path(run_root), Path(wheelhouse_dir)
    receipt = run_root / 'environment_receipt.json'
    if receipt.exists():
        raise FileExistsError('Preserve the existing environment attempt')
    expected = plan['expected_core_packages']
    names = set(expected) | {'torch', 'vllm', 'pip', 'packaging'}
    report = {'created_utc': datetime.now(timezone.utc).isoformat(),
              'scope': 'declared_public_comparison_environment_not_model_or_control_results',
              'mode': mode, 'status': 'started', 'before': cpu_runtime_snapshot(names),
              'after': None, 'wheel_manifest': [], 'incompatible_wheels': [],
              'selected_wheels_omitted_for_incompatibility': [], 'pip_return_code': None,
              'plan_sha256': environment_file_sha256(run_root / 'plan.json'),
              'model_inference_executed': False, 'fixture_controls_executed': False,
              'runtime_image_id_verified': False}
    def save():
        receipt.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        print('DECLARED_ENVIRONMENT=' + json.dumps({key: report.get(key) for key in
            ('mode', 'status', 'runtime_errors', 'incompatible_wheels', 'pip_return_code')}), flush=True)
        checkpoint_exporter(run_root)
    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise ValueError('Attach the complete official organizer wheelhouse')
        report['runtime_errors'] = cpu_runtime_errors(report['before'])
        report['wheel_manifest'] = classify_organizer_wheels(wheelhouse_dir.glob('*.whl'))
        selected = [row for row in report['wheel_manifest'] if row['selected']]
        report['incompatible_wheels'] = [row['original_name'] for row in selected if not row['compatible']]
        if not selected or report['runtime_errors'] or report['incompatible_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise RuntimeError('Use a compatible Linux x86_64 CPython 3.12 organizer worker; no incompatible wheels were omitted or installed')
        report['cutlass_hooks_removed'] = remove_recorded_cutlass_hooks()
        # Link staging stays outside the exported run; its wheel identities remain
        # in the receipt. It cannot collide with strict artifact link rejection.
        with tempfile.TemporaryDirectory(prefix='neural_swe_declared_wheels_') as temporary:
            links = Path(temporary)
            for row in selected:
                (links / row['install_name']).symlink_to(Path(row['source_path']).resolve())
            command = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                       '--no-deps', '--force-reinstall'] + [str(links / row['install_name']) for row in selected]
            report.update(status='installing', pip_command=command)
            save()
            result = subprocess.run(command, capture_output=True, text=True, timeout=1200)
            (run_root / 'organizer_install_stdout.txt').write_text(result.stdout or '', encoding='utf-8')
            (run_root / 'organizer_install_stderr.txt').write_text(result.stderr or '', encoding='utf-8')
            report['pip_return_code'] = result.returncode
        importlib.invalidate_caches()
        report['after'] = cpu_runtime_snapshot(names)
        if result.returncode != 0:
            raise RuntimeError('Organizer installation failed; retained install logs are available')
        if {name: report['after']['package_versions'][name] for name in expected} != expected:
            raise ValueError('Installed organizer core identity differs from the frozen plan')
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        report.update(declared_backend_check(mode, torch))
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        report['verify_task_sha256'] = verifier_sha
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Actual organizer verifier differs from the frozen contract')
        report['status'] = 'ready_for_' + mode
        save()
        return report
    except Exception as error:
        if report['status'] != 'preflight_rejected_before_install':
            report['status'] = 'setup_exception_before_experiment'
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / 'setup_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    if MODE == 'gpu_comparison':
        if not MODEL_PATH.is_dir():
            raise ValueError('Attach exact 31B QAT revision 2 model')
        for key, value in {'LITELLM_LOCAL_MODEL_COST_MAP':'True', 'TRANSFORMERS_NO_TF':'1',
            'VLLM_WORKER_MULTIPROC_METHOD':'spawn', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS':'1',
            'VLLM_ENGINE_READY_TIMEOUT_S':'1200', 'VLLM_NO_USAGE_STATS':'1',
            'OTEL_SDK_DISABLED':'true', 'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}.items():
            os.environ[key] = value
    prepare_declared_environment(WHEELHOUSE_DIR, RUN, PLAN, export_run, MODE)
    if hashlib.sha256((DATA_DIR / 'tasks.jsonl').read_bytes()).hexdigest() != PLAN['tasks_sha256']:
        raise ValueError('Official task revision differs from the frozen plan')
    if hashlib.sha256((DATA_DIR / 'HARNESS_README.md').read_bytes()).hexdigest() != PLAN['guide_sha256']:
        raise ValueError('Official guide revision differs from the frozen plan')
    from swegemma.models import load_tasks
    actual = {task.instance_id: task for task in load_tasks(DATA_DIR / 'tasks.jsonl')}
    for line in (RUN / 'agent_inputs.jsonl').read_text(encoding='utf-8').splitlines():
        problem = json.loads(line)
        if set(problem) != {'instance_id', 'repo', 'base_commit', 'problem_statement', 'hints_text'}:
            raise ValueError('Problem-only input includes unexpected fields')
        if any(getattr(actual[problem['instance_id']], name) != value for name, value in problem.items()):
            raise ValueError('Actual organizer problem-only input differs from frozen input')


In [ ]:
"""A small, locked development comparison using the actual organizer evaluator.

Development results cannot be reported as held-out or leaderboard results. Every
assigned arm/task pair remains in the denominator, including interrupted runs.
"""
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import re
import tempfile
import time


def assignment_key(row: dict) -> tuple:
    """Explicit trial IDs allow repeated assignments without merging outcomes."""
    values = (row['arm'], row['task_id'])
    for value in (*values, *([row['trial_id']] if 'trial_id' in row else [])):
        if not isinstance(value, str) or not re.fullmatch(r'[A-Za-z0-9_-]+', value):
            raise ValueError('Assignment identifiers must be safe nonempty path components')
    return (*values, row.get('trial_id'))


def atomic_json(path: Path, value: dict) -> None:
    """A crash must leave the previous complete receipt or the next complete one."""
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n',
                                         dir=path.parent, prefix='.' + path.name + '.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(json.dumps(value, indent=2) + '\n')
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()


def summarize(plan: dict, observations: list[dict]) -> dict:
    if not plan['schedule']:
        raise ValueError('A locked comparison must assign at least one trial')
    for arm in plan['arms']:
        assignment_key({'arm': arm, 'task_id': 'configuration'})
    expected = {assignment_key(row) for row in plan['schedule']}
    if len(expected) != len(plan['schedule']):
        raise ValueError('Duplicate planned assignment; repeats require distinct trial_id values')
    if any(pair[0] not in plan['arms'] for pair in expected):
        raise ValueError('Scheduled arm is not declared')
    seen, counts = set(), {arm: 0 for arm in plan['arms']}
    completed = 0
    for row in observations:
        key = assignment_key(row)
        if key not in expected or key in seen:
            raise ValueError('Unexpected or duplicate observed arm/task pair')
        seen.add(key)
        result = row.get('result', {})
        completed += int(result.get('status') == 'completed')
        passed = (result.get('status') == 'completed'
                  and result.get('official_resolved') is True
                  and result.get('test_exit_code') == 0
                  and result.get('patch_bytes', 0) > 0)
        counts[row['arm']] += int(passed)
    return {
        'scope': 'public_development_comparison_not_held_out_or_leaderboard',
        'assigned_pairs': len(expected), 'observed_pairs': len(seen),
        'completed_pairs': completed, 'incomplete_observed_pairs': len(seen) - completed,
        'unattempted_pairs': len(expected - seen),
        'arms': {arm: {'verified_resolved': counts[arm],
                      'assigned_count': sum(pair[0] == arm for pair in expected)}
                 for arm in plan['arms']},
        'official_leaderboard_score': None,
        'general_superiority_established': False,
        'interpretation': 'Development outcomes and repeated trials do not establish held-out general superiority.',
    }


def run_development(data_dir: Path, model_path: Path, run_root: Path, plan: dict) -> dict:
    # The notebook embeds the audited official_smoke.run definition first.
    from competition.official_smoke import run
    return execute_plan(run, data_dir, model_path, run_root, plan)


def execute_plan(runner, data_dir: Path, model_path: Path, run_root: Path, plan: dict,
                 *, checkpoint_exporter=None, session_started_monotonic=None) -> dict:
    destination = run_root / 'development_comparison.json'
    if destination.exists():
        raise FileExistsError('A comparison already exists in this run; preserve it and choose a new run directory')
    summarize(plan, [])  # Validate the entire schedule before invoking a model.
    walltime_budget = plan.get('notebook_walltime_budget_seconds')
    if walltime_budget is not None:
        if type(walltime_budget) is not int or walltime_budget <= 0 or session_started_monotonic is None:
            raise ValueError('A declared notebook walltime allowance requires its pre-setup timer')
    if hashlib.sha256((data_dir / 'tasks.jsonl').read_bytes()).hexdigest() != plan['tasks_sha256']:
        raise ValueError('Official task revision differs from the locked plan')
    for arm, identity in plan['arms'].items():
        agent_dir = run_root / 'agents' / arm
        if any(p.is_symlink() or not p.resolve().is_relative_to(run_root.resolve())
               for p in agent_dir.rglob('*')):
            raise ValueError('Agent configuration contains a link or escaped path: ' + arm)
        observed = {p.relative_to(agent_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in agent_dir.rglob('*') if p.is_file()}
        if observed != identity['files']:
            raise ValueError('Agent configuration differs from locked plan: ' + arm)
    observations, runtime = [], {}
    run_state = {'stopping_reason': None, 'notebook_walltime_budget_seconds': walltime_budget,
                 'session_timer_includes_setup': session_started_monotonic is not None}
    def save():
        report = {'created_utc': datetime.now(timezone.utc).isoformat(),
                  'plan_sha256': hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest(),
                  'observations': observations, **run_state, **summarize(plan, observations)}
        atomic_json(destination, report)
        return report
    save()
    if checkpoint_exporter is not None:
        checkpoint_exporter(run_root)
    for index, assignment in enumerate(plan['schedule']):
        arm, task_id = assignment['arm'], assignment['task_id']
        if walltime_budget is not None:
            remaining = session_started_monotonic + walltime_budget - time.monotonic()
            limits = plan['limits']
            required = limits['max_time_minutes'] * 60 + limits['timeout_seconds']
            required += 1200 if not runtime else 0  # Recorded server startup allowance.
            if remaining < required:
                run_state['stopping_reason'] = 'walltime_allowance_insufficient_for_next_assignment'
                return save()  # Never relabel the untouched slots as attempted model failures.
        observation = {'arm': arm, 'task_id': task_id,
                       'result': {'status': 'started',
                                  'started_utc': datetime.now(timezone.utc).isoformat(),
                                  'model_inference_executed': None}}
        if 'trial_id' in assignment:
            observation['trial_id'] = assignment['trial_id']
        observations.append(observation)
        save()
        print(f'DEVELOPMENT_START {index + 1}/{len(plan["schedule"])} {arm} {task_id}', flush=True)
        try:
            if checkpoint_exporter is not None:
                checkpoint_exporter(run_root)
            runner_kwargs = {'runtime': runtime}
            if 'limits' in plan:
                runner_kwargs['task_limits'] = plan['limits']
            result = runner(data_dir, model_path, run_root / 'agents' / arm,
                            run_root / 'results' / f'{index:02d}_{arm}_{task_id}', task_id,
                            **runner_kwargs)
            observation['result'] = result
            report = save()
            print('DEVELOPMENT_RESULT ' + json.dumps(observations[-1]), flush=True)
        except Exception as error:
            observation['result'] = {'status': 'infrastructure_exception',
                                     'error_type': type(error).__name__, 'error': str(error)}
            save()
            if checkpoint_exporter is not None:
                checkpoint_exporter(run_root)
            # Preserve unattempted slots; do not keep generating on a broken backend.
            raise
        if checkpoint_exporter is not None:
            checkpoint_exporter(run_root)
    return report


In [ ]:
def classify_fixture(clean: dict, reference: dict) -> str:
    if clean['test_exit_code'] == -1 or reference['test_exit_code'] == -1:
        return 'grading_setup_failed'
    if reference['resolved'] is not True or reference['test_exit_code'] != 0:
        return 'official_reference_not_verified'
    if clean['test_exit_code'] == 0:
        return 'clean_base_already_passes'
    if clean['test_exit_code'] in (1, 2):
        return 'clean_base_nonpassing_reference_verified'
    return 'unclassified_nonpassing_clean_base'


In [ ]:
"""Independent public CPU controls with reference and test text kept private."""
import asyncio
from concurrent.futures import ThreadPoolExecutor
from contextlib import redirect_stderr, redirect_stdout
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import io
import json
import logging
from pathlib import Path
import tempfile
import time



def sanitized_control_result(task_id, condition, patch, result, diagnostics):
    """Never serialize result objects, grader text, patches, or exception messages."""
    if condition not in {'clean_base', 'official_reference_patch'}:
        raise ValueError('Unknown control condition')
    if type(result.resolved) is not bool or type(result.test_exit_code) is not int:
        raise ValueError('Organizer control result has unexpected scalar types')
    return {'task_id': task_id, 'condition': condition, 'status': 'completed',
            'resolved': result.resolved, 'test_exit_code': result.test_exit_code,
            'duration_seconds': float(result.duration_seconds),
            'replayed_patch_sha256': hashlib.sha256(patch.encode('utf-8')).hexdigest(),
            'replayed_patch_bytes': len(patch.encode('utf-8')),
            'test_output_sha256': hashlib.sha256((result.test_output or '').encode('utf-8')).hexdigest(),
            'captured_diagnostic_sha256': hashlib.sha256(diagnostics.encode('utf-8')).hexdigest(),
            'reference_and_test_text_exported': False,
            'agent_generated_repair': False}


def run_public_fixture_controls(data_dir, run_root, plan, checkpoint_exporter):
    data_dir, run_root = Path(data_dir), Path(run_root)
    destination = run_root / 'fixture_controls.json'
    if destination.exists():
        raise FileExistsError('Preserve previous controls; use a new attempt')
    report = {'created_utc': datetime.now(timezone.utc).isoformat(), 'status': 'started',
              'scope': 'public_CPU_fixture_controls_not_agent_results_or_leaderboard',
              'model_inference_executed': False, 'agent_generated_success_count': None,
              'assigned_checks': 2 * len(plan['tasks']), 'observations': [], 'fixtures': [],
              'plan_sha256': hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest(),
              'reference_and_test_text_exported': False, 'official_leaderboard_score': None}
    def save():
        atomic_json(destination, report)
        checkpoint_exporter(run_root)
    save()
    try:
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        if torch.cuda.device_count() != 0:
            raise ValueError('CPU controls require Accelerator None')
        for name, expected in [('tasks.jsonl', plan['tasks_sha256']),
                               ('HARNESS_README.md', plan['guide_sha256'])]:
            if hashlib.sha256((data_dir / name).read_bytes()).hexdigest() != expected:
                raise ValueError('Organizer input identity differs from the frozen plan')
        packages = {name: importlib.metadata.version(name) for name in plan['expected_core_packages']}
        if packages != plan['expected_core_packages']:
            raise ValueError('Organizer package identity differs from the frozen plan')
        report['packages'] = packages
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Organizer verifier identity differs from the frozen plan')
        report['verify_task_sha256'] = verifier_sha
        actual_tasks = {task.instance_id: task for task in load_tasks(data_dir / 'tasks.jsonl')}
        selected_ids = {entry['instance_id'] for entry in plan['tasks']}
        # This access occurs only inside the independent CPU grader. Reference
        # bytes stay in memory and outside the exported artifact directory.
        raw_tasks = {row['instance_id']: row for row in map(json.loads,
                     (data_dir / 'tasks.jsonl').read_text().splitlines())
                     if row['instance_id'] in selected_ids}

        async def checks():
            for entry in plan['tasks']:
                task_id = entry['instance_id']
                task, raw_task = actual_tasks[task_id], raw_tasks[task_id]
                if any(raw_task[key] != entry[key] for key in ('instance_id', 'repo', 'base_commit')):
                    raise ValueError('Control task identity differs from the frozen plan')
                snapshot, base_snapshot, task_patch = apis['resolve_task_snapshot_paths'](
                    data_dir / 'snapshots', task_id, task.repo)
                outcomes = {}
                for condition in ('clean_base', 'official_reference_patch'):
                    row = {'task_id': task_id, 'condition': condition, 'status': 'started',
                           'reference_and_test_text_exported': False, 'agent_generated_repair': False}
                    report['observations'].append(row)
                    save()
                    patch = '' if condition == 'clean_base' else raw_task['patch']
                    captured = io.StringIO()
                    old_logging_disable = logging.root.manager.disable
                    try:
                        with tempfile.TemporaryDirectory(prefix='neural_swe_fixture_private_') as temporary:
                            scratch = Path(temporary)
                            (scratch / 'agent').mkdir()
                            config = EvalConfig(tasks_path=data_dir / 'tasks.jsonl',
                                snapshots_dir=data_dir / 'snapshots', results_dir=scratch / 'results',
                                submission_dir=scratch / 'agent', models={}, sandbox='subprocess',
                                timeout_seconds=plan['limits']['timeout_seconds'], wheels_dir=data_dir / 'wheels',
                                graph_dir=str(data_dir / 'graphs'), embeddings_dir=str(data_dir / 'embeddings'))
                            logging.disable(logging.CRITICAL)
                            with redirect_stdout(captured), redirect_stderr(captured):
                                evaluator = Evaluator(config)
                                result = await apis['verify_task'](evaluator.docker, config, task, snapshot,
                                    base_snapshot_path=base_snapshot, patch_path=task_patch,
                                    agent_patch=patch, start_time=time.perf_counter())
                        row.clear()
                        row.update(sanitized_control_result(task_id, condition, patch, result, captured.getvalue()))
                    except Exception as error:
                        row.update(status='infrastructure_exception', error_type=type(error).__name__,
                                   error_message_sha256=hashlib.sha256(str(error).encode()).hexdigest())
                        raise
                    finally:
                        logging.disable(old_logging_disable)
                        save()
                    outcomes[condition] = row
                    print('PUBLIC_FIXTURE_CONTROL=' + json.dumps(row), flush=True)
                report['fixtures'].append({'task_id': task_id, 'classification': classify_fixture(
                    outcomes['clean_base'], outcomes['official_reference_patch'])})
                save()
            report['status'] = 'completed'
            save()
            return report
        with ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(checks())).result()
    except Exception as error:
        report.update(status='infrastructure_exception', error_type=type(error).__name__,
                      error_message_sha256=hashlib.sha256(str(error).encode()).hexdigest())
        save()
        raise RuntimeError('CPU fixture controls stopped; inspect sanitized retained status and error type') from None


In [ ]:
"""Private grader diagnostics on an already exposed, fixed development panel."""
import asyncio
from concurrent.futures import ThreadPoolExecutor
from contextlib import redirect_stderr, redirect_stdout
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import io
import json
import logging
from pathlib import Path
import re
import tempfile
import time



def retain_private_grader_text(run_root, task_id, condition, result, reference_patch):
    """Retain grader text, never serialize the result, trace or reference patch."""
    if not re.fullmatch(r'(requests|fastapi)_\d+', task_id):
        raise ValueError('Unexpected diagnostic task identity')
    if condition not in {'clean_base', 'official_reference_patch'}:
        raise ValueError('Unexpected diagnostic condition')
    destination = Path(run_root) / 'private_diagnostics' / task_id / condition
    destination.mkdir(parents=True)
    text = result.test_output or ''
    error = getattr(result, 'error', None)
    if error is not None and not isinstance(error, str):
        raise ValueError('Unexpected grader error type')
    error = error or ''
    # A complete reference patch may be included in an application error. Its
    # bytes are never exported, even in this private diagnostic attempt.
    redacted = False
    if reference_patch:
        for value in (text, error):
            redacted = redacted or reference_patch in value
        text = text.replace(reference_patch, '[REFERENCE PATCH REDACTED]')
        error = error.replace(reference_patch, '[REFERENCE PATCH REDACTED]')
    rows = {}
    for name, value in [('test_output.txt', text), ('grader_error.txt', error)]:
        payload = value.encode('utf-8')
        target = destination / name
        target.write_bytes(payload)
        rows[name] = {'path': target.relative_to(run_root).as_posix(), 'bytes': len(payload),
                      'sha256': hashlib.sha256(payload).hexdigest()}
    return {'files': rows, 'complete_reference_patch_redacted': redacted,
            'validation_plaintext_exported_privately': True,
            'reference_patch_exported': False, 'agent_or_public_resource_input': False}


def retain_verifier_helpers(verify_task, run_root):
    module = inspect.getmodule(verify_task)
    names = ('verify_task', 'install_test_dependencies', 'setup_workspace_test_config',
             'install_editable_package', 'setup_container_wheels')
    rows = {}
    folder = Path(run_root) / 'organizer_sources'
    folder.mkdir()
    for name in names:
        function = verify_task if name == 'verify_task' else getattr(module, name, None)
        if function is None or not inspect.isfunction(function):
            rows[name] = {'available_in_loaded_verifier_namespace': False}
            continue
        payload = inspect.getsource(function).encode('utf-8')
        (folder / (name + '.py')).write_bytes(payload)
        rows[name] = {'available_in_loaded_verifier_namespace': True,
                      'sha256': hashlib.sha256(payload).hexdigest(), 'bytes': len(payload)}
    atomic_json(folder / 'sources.json', rows)
    return rows


def run_private_fixture_diagnostics(data_dir, run_root, plan, checkpoint_exporter):
    data_dir, run_root = Path(data_dir), Path(run_root)
    destination = run_root / 'private_fixture_diagnostics.json'
    if destination.exists():
        raise FileExistsError('Preserve this attempt; use a separately declared revision')
    report = {'created_utc': datetime.now(timezone.utc).isoformat(), 'status': 'started',
              'scope': 'private_exposed_development_grader_diagnostics_not_agent_repairs',
              'model_inference_executed': False, 'agent_generated_success_count': None,
              'assigned_checks': 2 * len(plan['tasks']), 'observations': [], 'fixtures': [],
              'plan_sha256': hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest(),
              'validation_plaintext_exported_privately': True, 'reference_patch_exported': False,
              'official_leaderboard_score': None, 'environment_modification_from_parent22': False}

    def save():
        atomic_json(destination, report)
        checkpoint_exporter(run_root)

    save()
    try:
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        if torch.cuda.device_count() != 0:
            raise ValueError('Private diagnostics require Accelerator None')
        for name, expected in [('tasks.jsonl', plan['tasks_sha256']),
                               ('HARNESS_README.md', plan['guide_sha256'])]:
            if hashlib.sha256((data_dir / name).read_bytes()).hexdigest() != expected:
                raise ValueError('Organizer input differs from the frozen diagnostic plan')
        packages = {name: importlib.metadata.version(name) for name in plan['expected_core_packages']}
        if packages != plan['expected_core_packages']:
            raise ValueError('Organizer packages differ from the frozen diagnostic plan')
        report['packages'] = packages
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Organizer verifier differs from the frozen diagnostic plan')
        report['verify_task_sha256'] = verifier_sha
        report['organizer_helper_sources'] = retain_verifier_helpers(apis['verify_task'], run_root)
        actual_tasks = {task.instance_id: task for task in load_tasks(data_dir / 'tasks.jsonl')}
        selected_ids = {entry['instance_id'] for entry in plan['tasks']}
        raw_tasks = {row['instance_id']: row for row in map(json.loads,
                     (data_dir / 'tasks.jsonl').read_text().splitlines())
                     if row['instance_id'] in selected_ids}

        async def checks():
            for entry in plan['tasks']:
                task_id = entry['instance_id']
                task, raw_task = actual_tasks[task_id], raw_tasks[task_id]
                if any(raw_task[key] != entry[key] for key in ('instance_id', 'repo', 'base_commit')):
                    raise ValueError('Diagnostic task identity differs from the frozen plan')
                snapshot, base_snapshot, task_patch = apis['resolve_task_snapshot_paths'](
                    data_dir / 'snapshots', task_id, task.repo)
                outcomes = {}
                for condition in ('clean_base', 'official_reference_patch'):
                    row = {'task_id': task_id, 'condition': condition, 'status': 'started',
                           'agent_generated_repair': False}
                    report['observations'].append(row)
                    save()
                    patch = '' if condition == 'clean_base' else raw_task['patch']
                    captured = io.StringIO()
                    old_logging_disable = logging.root.manager.disable
                    try:
                        with tempfile.TemporaryDirectory(prefix='neural_swe_private_diagnostic_') as temporary:
                            scratch = Path(temporary)
                            (scratch / 'agent').mkdir()
                            config = EvalConfig(tasks_path=data_dir / 'tasks.jsonl',
                                snapshots_dir=data_dir / 'snapshots', results_dir=scratch / 'results',
                                submission_dir=scratch / 'agent', models={}, sandbox='subprocess',
                                timeout_seconds=plan['limits']['timeout_seconds'], wheels_dir=data_dir / 'wheels',
                                graph_dir=str(data_dir / 'graphs'), embeddings_dir=str(data_dir / 'embeddings'))
                            logging.disable(logging.CRITICAL)
                            with redirect_stdout(captured), redirect_stderr(captured):
                                evaluator = Evaluator(config)
                                result = await apis['verify_task'](evaluator.docker, config, task, snapshot,
                                    base_snapshot_path=base_snapshot, patch_path=task_patch,
                                    agent_patch=patch, start_time=time.perf_counter())
                        row.clear()
                        row.update(sanitized_control_result(task_id, condition, patch, result, captured.getvalue()))
                        row['private_diagnostics'] = retain_private_grader_text(
                            run_root, task_id, condition, result, raw_task['patch'])
                        row['reference_and_test_text_exported'] = True
                    except Exception as error:
                        row.update(status='infrastructure_exception', error_type=type(error).__name__,
                                   error_message_sha256=hashlib.sha256(str(error).encode()).hexdigest())
                        raise
                    finally:
                        logging.disable(old_logging_disable)
                        save()
                    outcomes[condition] = row
                    print('PRIVATE_FIXTURE_DIAGNOSTIC=' + json.dumps(row), flush=True)
                report['fixtures'].append({'task_id': task_id, 'classification': classify_fixture(
                    outcomes['clean_base'], outcomes['official_reference_patch'])})
                save()
            report['status'] = 'completed'
            save()
            return report

        with ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(checks())).result()
    except Exception as error:
        report.update(status='infrastructure_exception', error_type=type(error).__name__,
                      error_message_sha256=hashlib.sha256(str(error).encode()).hexdigest())
        save()
        raise RuntimeError('Private diagnostic stopped; inspect retained scalar receipt') from None


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    result = run_private_fixture_diagnostics(DATA_DIR, RUN, PLAN, export_run)
    print('PRIVATE_FIXTURE_DIAGNOSTICS=' + json.dumps(result))
else:
    print('Definitions only; no grader text, references or model accessed')


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    export_run(RUN)
else:
    print('No worker evidence exported by local definition check')
